# 03 — Modelo de Recomendação

## 1. Definição da abordagem

Esta etapa tem como objetivo desenvolver uma prova de conceito (POC) de um sistema híbrido de recomendação de cursos da Escola Virtual de Governo (EV.G).

A abordagem proposta combina duas fontes de informação:

- **Content-Based Filtering:** utiliza as características dos cursos, como nome, eixos temáticos, apresentação, público-alvo e conteúdo programático, para identificar cursos semelhantes aos interesses demonstrados pelo usuário.

- **Collaborative Filtering:** utiliza o histórico de interações entre usuários e cursos para identificar padrões de comportamento e gerar recomendações a partir dessas relações.

Os resultados dos dois componentes serão posteriormente combinados em uma abordagem híbrida.

Como a base não possui avaliações explícitas dos cursos, a existência de uma interação usuário–curso será utilizada como feedback implícito binário. Os diferentes status das matrículas são preservados na base, mas não recebem pesos positivos ou negativos nesta primeira POC, evitando a introdução de ponderações arbitrárias. Múltiplas interações entre o mesmo usuário e curso são consolidadas em uma única relação.

A avaliação da POC também deverá considerar a dimensão temporal das interações, evitando que informações posteriores ao período de teste sejam utilizadas durante o treinamento.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pasta_dados = Path("../dados/processados")

arquivo_interacoes = pasta_dados / "interacoes_cursos_atuais.parquet"
arquivo_catalogo = pasta_dados / "catalogo_canonico.parquet"

con = duckdb.connect()

## 2. Construção do feedback implícito

### 2.1 Interpretação das interações

A base da EV.G não possui avaliações explícitas dos usuários sobre os cursos. Dessa forma, a matrícula é utilizada como evidência de interesse implícito, pois representa uma escolha realizada pelo usuário.

O status da matrícula, entretanto, não é interpretado diretamente como uma avaliação positiva ou negativa. Estados como desistência, reprovação ou trancamento podem estar associados a diferentes circunstâncias e não necessariamente representam ausência de interesse pelo conteúdo.

A análise exploratória também identificou usuários que interagiram mais de uma vez com o mesmo curso e apresentaram diferentes status ao longo do tempo. Entre os pares usuário-curso que possuíam exatamente registros de desistência e conclusão, a desistência ocorreu antes da conclusão na maioria dos casos analisados.

Por esse motivo, o status será utilizado como informação complementar sobre a trajetória da interação, enquanto a existência da matrícula constitui o sinal básico de interesse implícito.

As múltiplas matrículas de um mesmo usuário em um mesmo curso serão consolidadas em uma única relação usuário-curso, preservando informações relevantes sobre seu histórico de interação.

### 2.2 Estratégia de divisão temporal

Para avaliar o sistema de recomendação sem utilizar informações futuras durante o treinamento, será adotada uma divisão cronológica dos dados.

As interações serão separadas em três períodos:

- **Treinamento:** interações registradas até 31 de dezembro de 2024;
- **Validação:** interações registradas entre 1º de janeiro e 31 de dezembro de 2025;
- **Teste:** interações registradas entre 1º de janeiro e 2 de setembro de 2026, correspondente ao período disponível na base.

O conjunto de treinamento será utilizado para construção dos componentes Content-Based e Collaborative Filtering. O período de validação será utilizado para avaliar alternativas de configuração e definir parâmetros do modelo híbrido, como a combinação dos scores produzidos pelos dois componentes.

Após a definição da configuração da POC, o período de 2026 será utilizado como conjunto de teste final, permitindo avaliar o comportamento do modelo em interações posteriores às utilizadas durante seu desenvolvimento.

Embora os dados de 2026 não representem o ano completo, isso não impede sua utilização como conjunto de teste, pois a avaliação considera as interações efetivamente observadas no período disponível e não realiza comparação direta entre totais anuais.

A separação temporal também reduz o risco de vazamento de informação, garantindo que interações futuras não sejam utilizadas para prever comportamentos anteriores.

### 2.3 Distribuição das interações por período

Antes da construção dos conjuntos utilizados na modelagem, foi verificada a quantidade de interações, usuários e cursos disponíveis em cada período da divisão temporal.

Essa análise permite avaliar se os conjuntos de treinamento, validação e teste possuem volume de dados suficiente para o desenvolvimento e a avaliação da POC.

In [9]:
distribuicao_periodos = con.execute(f"""
    SELECT
        CASE
            WHEN dt_matricula < DATE '2025-01-01'
                THEN 'Treino'
            WHEN dt_matricula < DATE '2026-01-01'
                THEN 'Validação'
            ELSE 'Teste'
        END AS conjunto,

        COUNT(*) AS qtd_interacoes,
        COUNT(DISTINCT codigo_pessoa) AS qtd_usuarios,
        COUNT(DISTINCT id_curso_canonico) AS qtd_cursos

    FROM read_parquet('{arquivo_interacoes.as_posix()}')

    GROUP BY conjunto

    ORDER BY
        CASE conjunto
            WHEN 'Treino' THEN 1
            WHEN 'Validação' THEN 2
            WHEN 'Teste' THEN 3
        END
""").df()

distribuicao_periodos

,conjunto,qtd_interacoes,qtd_usuarios,qtd_cursos
0,Treino,13104833,4270327,788
1,Validação,4634436,1939000,975
2,Teste,3623733,1626454,1029


#### Interpretação

A divisão temporal resultou em conjuntos com volume expressivo de interações e usuários, permitindo o desenvolvimento e a avaliação da POC.

Observou-se, entretanto, uma diferença importante na quantidade de cursos presentes em cada período. Enquanto o conjunto de treinamento contém interações com 788 cursos canônicos, a validação apresenta 975 cursos e o teste 1.029 dos 1.030 cursos existentes no catálogo atual.

Esse resultado indica que parte dos cursos presentes nos períodos de validação e teste não possui histórico de interação no conjunto de treinamento. Essa situação caracteriza um cenário de *item cold-start* para o componente colaborativo e será considerada na avaliação do sistema híbrido.

### 2.4 Cobertura temporal dos cursos

Como cursos sem histórico no treinamento podem surgir após o período utilizado para treinamento, foi analisada a cobertura dos cursos de validação e teste em relação aos cursos que possuem histórico no conjunto de treinamento.

Essa análise é especialmente relevante para o componente Collaborative Filtering, que depende de interações históricas para estabelecer relações entre os itens.

In [10]:
cobertura_cursos = con.execute(f"""
    WITH cursos_treino AS (
        SELECT DISTINCT id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula < DATE '2025-01-01'
    ),

    cursos_validacao AS (
        SELECT DISTINCT id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula >= DATE '2025-01-01'
          AND dt_matricula < DATE '2026-01-01'
    ),

    cursos_teste AS (
        SELECT DISTINCT id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula >= DATE '2026-01-01'
    )

    SELECT
        'Validação' AS conjunto,
        COUNT(*) AS total_cursos,
        COUNT(*) FILTER (
            WHERE id_curso_canonico IN (
                SELECT id_curso_canonico FROM cursos_treino
            )
        ) AS cursos_conhecidos,
        COUNT(*) FILTER (
            WHERE id_curso_canonico NOT IN (
                SELECT id_curso_canonico FROM cursos_treino
            )
        ) AS cursos_novos
    FROM cursos_validacao

    UNION ALL

    SELECT
        'Teste',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE id_curso_canonico IN (
                SELECT id_curso_canonico FROM cursos_treino
            )
        ),
        COUNT(*) FILTER (
            WHERE id_curso_canonico NOT IN (
                SELECT id_curso_canonico FROM cursos_treino
            )
        )
    FROM cursos_teste
""").df()

cobertura_cursos

,conjunto,total_cursos,cursos_conhecidos,cursos_novos
0,Validação,975,788,187
1,Teste,1029,787,242


#### Interpretação

Todos os 788 cursos presentes no conjunto de treinamento também aparecem na validação, enquanto 787 deles aparecem novamente no período de teste.

Além disso, foram identificados 187 cursos na validação e 242 cursos no teste sem histórico de interação no conjunto de treinamento. Esses itens representam aproximadamente 19,2% e 23,5% dos cursos disponíveis em cada período, respectivamente.

Para o componente Collaborative Filtering, esses cursos representam um problema de *item cold-start*, pois não existem interações anteriores ao corte temporal a partir das quais suas relações comportamentais possam ser aprendidas. O componente Content-Based, por outro lado, poderá utilizar os atributos disponíveis no catálogo para representar esses itens.

Entretanto, a quantidade de cursos novos, isoladamente, não determina seu impacto sobre a avaliação. Por isso, também será analisada a proporção das interações de validação e teste associada a cursos sem histórico no conjunto de treinamento.

### 2.5 Impacto dos cursos sem histórico nas interações futuras

Além da quantidade de cursos sem histórico no treinamento, foi analisada a participação desses itens nas interações observadas nos períodos de validação e teste.

Essa análise permite dimensionar o impacto do cenário de *item cold-start* sobre a avaliação do sistema.

In [11]:
impacto_cold_start = con.execute(f"""
    WITH cursos_treino AS (
        SELECT DISTINCT id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula < DATE '2025-01-01'
    ),

    periodos AS (
        SELECT
            CASE
                WHEN dt_matricula >= DATE '2025-01-01'
                 AND dt_matricula < DATE '2026-01-01'
                    THEN 'Validação'
                WHEN dt_matricula >= DATE '2026-01-01'
                    THEN 'Teste'
            END AS conjunto,
            id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula >= DATE '2025-01-01'
    )

    SELECT
        conjunto,
        COUNT(*) AS total_interacoes,

        COUNT(*) FILTER (
            WHERE id_curso_canonico IN (
                SELECT id_curso_canonico FROM cursos_treino
            )
        ) AS interacoes_cursos_conhecidos,

        COUNT(*) FILTER (
            WHERE id_curso_canonico NOT IN (
                SELECT id_curso_canonico FROM cursos_treino
            )
        ) AS interacoes_cursos_novos,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE id_curso_canonico NOT IN (
                    SELECT id_curso_canonico FROM cursos_treino
                )
            ) / COUNT(*),
            2
        ) AS percentual_cold_start

    FROM periodos
    GROUP BY conjunto

    ORDER BY
        CASE conjunto
            WHEN 'Validação' THEN 1
            WHEN 'Teste' THEN 2
        END
""").df()

impacto_cold_start

,conjunto,total_interacoes,interacoes_cursos_conhecidos,interacoes_cursos_novos,percentual_cold_start
0,Validação,4634436,3734333,900103,19.42
1,Teste,3623733,2504597,1119136,30.88


#### Interpretação

O cenário de *item cold-start* apresenta impacto relevante sobre as interações futuras. Na validação, 900.103 interações (19,42%) correspondem a cursos sem histórico no conjunto de treinamento. No período de teste, essa quantidade aumenta para 1.119.136 interações, representando 30,88% das interações observadas.

Esses resultados evidenciam uma limitação importante de um recomendador exclusivamente colaborativo: cursos que não possuem histórico no período de treinamento não podem ter suas relações comportamentais aprendidas a partir das interações anteriores.

Nesse cenário, a abordagem híbrida torna-se especialmente relevante. Enquanto o componente Collaborative Filtering utiliza os padrões históricos de interação para os cursos já conhecidos, o componente Content-Based pode representar cursos sem histórico de interação no treinamento a partir de seus atributos e conteúdos disponíveis no catálogo.

Por esse motivo, além da avaliação global do sistema, os resultados da POC poderão ser analisados separadamente para itens conhecidos e itens em situação de *cold-start*.

### 2.6 Verificação da data de lançamento dos cursos

Foi considerada a possibilidade de utilizar a variável `data_lancamento` do catálogo para distinguir cursos efetivamente lançados após o período de treinamento daqueles que apenas não apresentavam interações anteriores.

Entretanto, durante a inspeção da variável foram identificadas datas posteriores ao período máximo disponível na base de interações, incluindo registros referentes aos anos de 2027, 2029 e 2030, enquanto o histórico analisado possui registros somente até setembro de 2026.

Diante dessa inconsistência, a variável `data_lancamento` não será utilizada, nesta POC, para determinar a disponibilidade temporal dos cursos.

Assim, os itens ausentes do conjunto de treinamento serão tratados de forma mais precisa como **cursos sem histórico de interação no treinamento**, condição suficiente para caracterizar uma situação de *item cold-start* do ponto de vista do componente Collaborative Filtering, sem assumir que esses cursos necessariamente foram lançados após o corte temporal.

In [12]:
amostra_datas_lancamento = con.execute(f"""
    SELECT
        id_curso_canonico,
        nome_curso,
        data_lancamento
    FROM read_parquet('{arquivo_catalogo.as_posix()}')
    LIMIT 15
""").df()

amostra_datas_lancamento

,id_curso_canonico,nome_curso,data_lancamento
0,1,Estudo Técnico Preliminar Avançado para Contra...,2014-05-21 00:00:00
1,2,Scrum en el Contexto del Servicio Público,2015-10-21 00:00:00
2,3,Documentos fiscais aplicados à vigilância e re...,2027-06-22 00:00:00
3,4,Abordagem Lean aplicada à Transformação Digita...,2020-04-22 00:00:00
4,5,"Reorganização Societária, blindagem patrimonia...",2030-08-22 00:00:00
5,6,Inclusão de gênero na ponta da língua,2006-03-23 00:00:00
6,7,"Eixo 4 - Inteligência Artificial, Machine Lear...",2024-09-25 00:00:00
7,8,Ética Profissional do Serviço Público na Recei...,2013-05-20 00:00:00
8,9,Obras Públicas: contratação e fiscalização sob...,2012-01-26 00:00:00
9,10,A Responsabilização Administrativa na Lei Anti...,2026-01-24 00:00:00


### 2.7 Cobertura temporal dos usuários

Para que uma recomendação personalizada seja avaliada, o usuário precisa possuir histórico de interação anterior ao período utilizado como alvo.

Na etapa de validação, portanto, serão considerados usuários com interações anteriores a 2025 que também apresentem interações em 2025. Para o teste final, após a definição dos parâmetros e o retreinamento do modelo com os dados disponíveis até o final de 2025, serão considerados usuários com histórico anterior a 2026 que também apresentem interações em 2026.

Usuários que aparecem pela primeira vez no próprio período de avaliação representam um cenário de *user cold-start* e não podem ser avaliados pelo mesmo protocolo de recomendação personalizada baseada em histórico.

In [13]:
cobertura_usuarios = con.execute(f"""
    WITH primeira_interacao AS (
        SELECT
            codigo_pessoa,
            MIN(dt_matricula) AS dt_primeira_interacao
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        GROUP BY codigo_pessoa
    ),

    usuarios_validacao AS (
        SELECT DISTINCT codigo_pessoa
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula >= DATE '2025-01-01'
          AND dt_matricula < DATE '2026-01-01'
    ),

    usuarios_teste AS (
        SELECT DISTINCT codigo_pessoa
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula >= DATE '2026-01-01'
    )

    SELECT
        'Validação' AS conjunto,
        COUNT(*) AS total_usuarios,
        COUNT(*) FILTER (
            WHERE p.dt_primeira_interacao < DATE '2025-01-01'
        ) AS usuarios_com_historico,
        COUNT(*) FILTER (
            WHERE p.dt_primeira_interacao >= DATE '2025-01-01'
        ) AS usuarios_sem_historico

    FROM usuarios_validacao v
    INNER JOIN primeira_interacao p
        ON v.codigo_pessoa = p.codigo_pessoa

    UNION ALL

    SELECT
        'Teste',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE p.dt_primeira_interacao < DATE '2026-01-01'
        ),
        COUNT(*) FILTER (
            WHERE p.dt_primeira_interacao >= DATE '2026-01-01'
        )

    FROM usuarios_teste t
    INNER JOIN primeira_interacao p
        ON t.codigo_pessoa = p.codigo_pessoa
""").df()

cobertura_usuarios

,conjunto,total_usuarios,usuarios_com_historico,usuarios_sem_historico
0,Validação,1939000,704659,1234341
1,Teste,1626454,686410,940044


#### Interpretação

A análise mostrou uma presença expressiva de usuários sem histórico anterior aos períodos de avaliação.

Na validação, 704.659 usuários (36,34%) possuem pelo menos uma interação anterior a 2025, enquanto 1.234.341 (63,66%) aparecem pela primeira vez na base durante o próprio período de validação.

No teste, após considerar 2025 como parte do histórico disponível para o retreinamento, 686.410 usuários (42,20%) possuem interações anteriores a 2026, enquanto 940.044 (57,80%) aparecem pela primeira vez no período de teste.

Como os componentes personalizados da POC dependem de algum histórico do usuário para construir suas recomendações, a avaliação principal será realizada sobre os usuários que possuem histórico anterior ao respectivo período de avaliação. Os usuários sem histórico serão tratados separadamente como cenário de *user cold-start*.

### 2.8 Definição dos itens relevantes para avaliação

Para a avaliação do recomendador, serão considerados como itens relevantes os cursos distintos com os quais o usuário interagiu no período futuro e que ainda não faziam parte de seu histórico anterior.

Dessa forma, repetições de cursos já realizados anteriormente pelo mesmo usuário não serão consideradas novos alvos de recomendação.

Essa definição aproxima o protocolo de avaliação do objetivo do sistema: recomendar novos cursos potencialmente relevantes a partir do histórico conhecido do usuário.

In [14]:
alvos_validacao = con.execute(f"""
    WITH historico AS (
        SELECT DISTINCT
            codigo_pessoa,
            id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula < DATE '2025-01-01'
    ),

    interacoes_2025 AS (
        SELECT DISTINCT
            codigo_pessoa,
            id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula >= DATE '2025-01-01'
          AND dt_matricula < DATE '2026-01-01'
    )

    SELECT
        v.codigo_pessoa,
        COUNT(*) AS qtd_itens_relevantes
    FROM interacoes_2025 v

    INNER JOIN (
        SELECT DISTINCT codigo_pessoa
        FROM historico
    ) u
        ON v.codigo_pessoa = u.codigo_pessoa

    LEFT JOIN historico h
        ON v.codigo_pessoa = h.codigo_pessoa
       AND v.id_curso_canonico = h.id_curso_canonico

    WHERE h.id_curso_canonico IS NULL

    GROUP BY v.codigo_pessoa
""").df()

alvos_validacao["qtd_itens_relevantes"].describe()

count    645919.000000
mean          2.848560
std           5.358217
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max         744.000000
Name: qtd_itens_relevantes, dtype: float64

#### Interpretação

Dos 704.659 usuários da validação que possuem histórico anterior a 2025, 645.919 apresentam pelo menos um curso distinto em 2025 que ainda não fazia parte de seu histórico, correspondendo a aproximadamente 91,67% desses usuários.

Entre os usuários avaliáveis, a mediana é de 2 novos cursos relevantes por usuário e 75% apresentam até 3 itens relevantes. A média de 2,85 itens é superior à mediana devido à presença de usuários com quantidade elevada de novos cursos, chegando ao máximo de 744.

Dessa forma, a avaliação será baseada no conjunto de novos cursos observados para cada usuário, em vez da seleção arbitrária de um único item futuro.

### 2.9 Métricas de avaliação

A avaliação da POC será realizada a partir de listas de recomendação Top-K, comparando os cursos recomendados com os novos cursos efetivamente observados para cada usuário no período futuro.

Inicialmente será utilizado **K = 10**, representando uma lista com os dez cursos mais recomendados para cada usuário.

Serão consideradas principalmente as seguintes métricas:

- **Recall@10:** proporção dos itens relevantes de cada usuário que aparecem entre as dez recomendações;
- **Hit Rate@10:** proporção de usuários para os quais pelo menos um item relevante aparece entre as dez recomendações;
- **NDCG@10:** métrica que considera tanto a presença dos itens relevantes quanto sua posição no ranking, atribuindo maior importância aos acertos apresentados nas primeiras posições.

As métricas serão calculadas sobre usuários com histórico anterior e pelo menos um novo curso observado no período de avaliação. Quando pertinente, os resultados também serão analisados separadamente entre cursos com e sem histórico no conjunto utilizado para treinamento.

In [15]:
import sklearn

print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.8.0


## 3. Componente Content-Based

O componente Content-Based será responsável por representar os cursos a partir de suas próprias características, permitindo identificar similaridades entre os conteúdos independentemente do histórico coletivo de interações.

Essa abordagem é especialmente relevante para o presente projeto devido à existência de cursos sem histórico de interação no período de treinamento, situação em que o componente Collaborative Filtering não dispõe de informações comportamentais suficientes.

Para a representação dos cursos serão utilizados, inicialmente, os campos textuais do catálogo que apresentaram maior disponibilidade e conteúdo informativo durante a análise exploratória.

### 3.1 Preparação dos atributos textuais

Para a construção da representação dos cursos foram selecionados os campos `nome_curso`, `eixos_tematicos`, `competencias`, `apresentacao`, `publico_alvo` e `conteudo_programatico`.

Esses atributos descrevem diferentes aspectos dos cursos, como tema, conteúdo, público e competências abordadas. Campos com baixa variabilidade ou de natureza numérica não serão incorporados diretamente à representação textual nesta primeira versão da POC.

Os valores ausentes serão substituídos por texto vazio antes da concatenação dos atributos.

In [16]:
catalogo_modelo = con.execute(f"""
    SELECT
        id_curso_canonico,
        nome_curso,
        eixos_tematicos,
        competencias,
        apresentacao,
        publico_alvo,
        conteudo_programatico
    FROM read_parquet('{arquivo_catalogo.as_posix()}')
    ORDER BY id_curso_canonico
""").df()

catalogo_modelo.shape

(1030, 7)

### 3.2 Construção da representação textual dos cursos

Os atributos selecionados foram combinados em uma única representação textual para cada curso.

Nesta configuração inicial da POC, o nome do curso e os eixos temáticos recebem maior ênfase na representação, por sintetizarem diretamente o assunto principal do item. Os demais campos complementam essa representação com informações sobre competências, apresentação, público-alvo e conteúdo programático.

Os pesos utilizados nesta etapa constituem uma configuração inicial e poderão ser ajustados posteriormente com base nos resultados obtidos no conjunto de validação.

In [17]:
colunas_texto = [
    "nome_curso",
    "eixos_tematicos",
    "competencias",
    "apresentacao",
    "publico_alvo",
    "conteudo_programatico"
]

catalogo_modelo[colunas_texto] = (
    catalogo_modelo[colunas_texto]
    .fillna("")
    .astype(str)
)

catalogo_modelo["texto_combinado"] = (
    catalogo_modelo["nome_curso"] + " "
    + catalogo_modelo["nome_curso"] + " "
    + catalogo_modelo["nome_curso"] + " "
    + catalogo_modelo["eixos_tematicos"] + " "
    + catalogo_modelo["eixos_tematicos"] + " "
    + catalogo_modelo["competencias"] + " "
    + catalogo_modelo["apresentacao"] + " "
    + catalogo_modelo["publico_alvo"] + " "
    + catalogo_modelo["conteudo_programatico"]
)

catalogo_modelo[
    ["id_curso_canonico", "nome_curso", "texto_combinado"]
].head()

,id_curso_canonico,nome_curso,texto_combinado
0,1,Estudo Técnico Preliminar Avançado para Contra...,Estudo Técnico Preliminar Avançado para Contra...
1,2,Scrum en el Contexto del Servicio Público,Scrum en el Contexto del Servicio Público Scru...
2,3,Documentos fiscais aplicados à vigilância e re...,Documentos fiscais aplicados à vigilância e re...
3,4,Abordagem Lean aplicada à Transformação Digita...,Abordagem Lean aplicada à Transformação Digita...
4,5,"Reorganização Societária, blindagem patrimonia...","Reorganização Societária, blindagem patrimonia..."


### 3.3 Vetorização TF-IDF

A representação textual dos cursos foi convertida em vetores numéricos por meio da técnica TF-IDF (*Term Frequency–Inverse Document Frequency*).

O TF-IDF atribui maior importância aos termos que são representativos de determinado documento, mas menos frequentes no conjunto completo de documentos, permitindo destacar palavras que contribuem para diferenciar os cursos.

Nesta primeira configuração, serão consideradas palavras individuais e combinações de duas palavras (*unigrams* e *bigrams*), permitindo representar tanto termos isolados quanto expressões relevantes presentes nas descrições dos cursos.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

matriz_tfidf = tfidf.fit_transform(
    catalogo_modelo["texto_combinado"]
)

print("Dimensão da matriz TF-IDF:", matriz_tfidf.shape)
print("Quantidade de termos:", len(tfidf.get_feature_names_out()))

Dimensão da matriz TF-IDF: (1030, 14526)
Quantidade de termos: 14526


### 3.4 Similaridade entre os cursos

Após a vetorização, foi utilizada a similaridade cosseno para comparar as representações TF-IDF dos cursos.

A similaridade cosseno mede a proximidade entre dois vetores considerando sua orientação no espaço vetorial. Quanto maior o valor obtido, maior a similaridade entre as representações textuais dos cursos.

Como o catálogo canônico possui 1.030 cursos, é possível calcular a matriz completa de similaridade item-item sem necessidade de comparar diretamente os milhões de usuários existentes na base.

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

similaridade_content = cosine_similarity(matriz_tfidf)

print("Dimensão da matriz de similaridade:", similaridade_content.shape)

Dimensão da matriz de similaridade: (1030, 1030)


In [21]:
catalogo_modelo.loc[
    catalogo_modelo["nome_curso"].str.contains(
        "dados|python|ciência de dados",
        case=False,
        na=False
    ),
    ["id_curso_canonico", "nome_curso"]
].head(20)

,id_curso_canonico,nome_curso
6,7,"Eixo 4 - Inteligência Artificial, Machine Lear..."
22,23,LGPD: Como coordenar a atuação do município pa...
23,24,Eixo 3 - Compartilhamento de Dados Programa - ...
33,34,Introdução à Ciência de Dados - Tratamento de ...
44,45,Gerenciar Dados com o Microsoft 365
51,52,Uso de Dados Raciais Aplicados às Políticas Pú...
53,54,Eixo 2 - Fundamentos de Ciência de Dados Progr...
91,92,Eixo 2 - Atuando na governança e na aplicação ...
102,103,Planos de Adaptação: dados espaciais e indicad...
107,108,Fundamentos da Lei Geral de Proteção de Dados ...


### 3.5 Inspeção das similaridades geradas

Antes da utilização do componente Content-Based na recomendação personalizada, foi realizada uma inspeção qualitativa das similaridades produzidas pelo modelo.

Para isso, foi selecionado um curso do catálogo e identificados os cursos com maior similaridade cosseno em relação à sua representação TF-IDF. Essa etapa permite verificar se as relações encontradas pelo modelo apresentam coerência temática antes de sua aplicação aos perfis dos usuários.

In [22]:
id_curso_teste = 34

indice_curso = catalogo_modelo.index[
    catalogo_modelo["id_curso_canonico"] == id_curso_teste
][0]

scores = similaridade_content[indice_curso]

indices_similares = scores.argsort()[::-1]

indices_similares = [
    i for i in indices_similares
    if i != indice_curso
][:10]

resultado_similares = catalogo_modelo.loc[
    indices_similares,
    ["id_curso_canonico", "nome_curso"]
].copy()

resultado_similares["similaridade"] = scores[indices_similares]

resultado_similares

,id_curso_canonico,nome_curso,similaridade
153,154,Introdução à Ciência de Dados - Modelos de Agr...,0.415848
837,838,Introdução à Ciência de Dados - Modelos de Reg...,0.348671
627,628,Introdução à Ciência de Dados - Modelos de Cla...,0.338269
263,264,Introdução à Ciência de Dados - Estatística Es...,0.337595
577,578,Introdução à Ciência de Dados - Descoberta de ...,0.324585
729,730,Introdução à Ciência de Dados - Redes Complexas,0.314378
382,383,Introdução à Ciência de Dados - Conjuntos Freq...,0.301075
981,982,Introdução à Ciência de Dados - Conceitos e Fe...,0.267548
300,301,Análise de Dados em Linguagem R,0.193354
132,133,Capacitação em Governança de Dados,0.165738


#### Interpretação

A inspeção qualitativa indicou coerência nas similaridades produzidas pelo componente Content-Based.

Utilizando como referência o curso **“Introdução à Ciência de Dados - Tratamento de Dados”**, os maiores valores de similaridade foram atribuídos predominantemente a outros cursos da série “Introdução à Ciência de Dados”, incluindo conteúdos relacionados a agrupamento, regressão, classificação, estatística e outros tópicos da área.

Também foram identificados, entre os dez cursos mais semelhantes, conteúdos relacionados à análise e à governança de dados.

O resultado indica que a representação TF-IDF, combinada à similaridade cosseno, é capaz de identificar relações temáticas coerentes entre os cursos do catálogo. Essa inspeção constitui uma verificação qualitativa inicial; o desempenho do recomendador será posteriormente avaliado quantitativamente sobre as interações futuras dos usuários.

### 3.6 Construção do perfil de interesse do usuário

Para produzir recomendações personalizadas, o histórico de cursos de cada usuário será utilizado para construir uma representação de seus interesses.

O perfil será obtido a partir da combinação das representações TF-IDF dos cursos presentes em seu histórico anterior ao período de avaliação. Esse perfil poderá então ser comparado às representações dos demais cursos do catálogo por meio da similaridade cosseno.

Os cursos já presentes no histórico do usuário serão excluídos da lista de candidatos, uma vez que o objetivo da POC é recomendar novos cursos.

In [23]:
usuario_exemplo = con.execute(f"""
    SELECT
        codigo_pessoa,
        COUNT(DISTINCT id_curso_canonico) AS qtd_cursos
    FROM read_parquet('{arquivo_interacoes.as_posix()}')
    WHERE dt_matricula < DATE '2025-01-01'
    GROUP BY codigo_pessoa
    HAVING COUNT(DISTINCT id_curso_canonico) BETWEEN 5 AND 10
    ORDER BY RANDOM()
    LIMIT 1
""").df()

usuario_exemplo

,codigo_pessoa,qtd_cursos
0,4f1921cbb0,6


In [24]:
historico_exemplo = con.execute(f"""
    SELECT DISTINCT
        i.id_curso_canonico,
        i.nome_curso
    FROM read_parquet('{arquivo_interacoes.as_posix()}') i
    WHERE i.codigo_pessoa = '4f1921cbb0'
      AND i.dt_matricula < DATE '2025-01-01'
    ORDER BY i.nome_curso
""").df()

historico_exemplo

,id_curso_canonico,nome_curso
0,324,A Previdência Social dos Servidores Públicos: ...
1,326,Gestão da Estratégia com BSC - Fundamentos
2,873,Gestão da Informação e Documentação - Conceito...
3,946,Gestão e preservação de documentos digitais
4,346,Nova lei de licitações: Visão Geral
5,606,Ética e Serviço Público


In [26]:
ids_historico = historico_exemplo["id_curso_canonico"].tolist()

indices_historico = catalogo_modelo.index[
    catalogo_modelo["id_curso_canonico"].isin(ids_historico)
].tolist()

perfil_usuario = np.asarray(
    matriz_tfidf[indices_historico].mean(axis=0)
)

scores_usuario = cosine_similarity(
    perfil_usuario,
    matriz_tfidf
).flatten()

scores_usuario[indices_historico] = -1

indices_recomendados = scores_usuario.argsort()[::-1][:10]

recomendacoes_exemplo = catalogo_modelo.loc[
    indices_recomendados,
    ["id_curso_canonico", "nome_curso"]
].copy()

recomendacoes_exemplo["score_content"] = (
    scores_usuario[indices_recomendados]
)

recomendacoes_exemplo

,id_curso_canonico,nome_curso,score_content
492,493,Noções Básicas de Gerenciamento em Serviços Ar...,0.284830
800,801,Elaboração de instrumentos de gestão de docume...,0.267291
546,547,Gestão Documental,0.260675
246,247,"Arquivos Permanentes: recolhimento, processame...",0.253500
769,770,Introdução às Práticas Arquivísticas,0.252638
920,921,Procedimentos de Protocolo,0.241938
210,211,Previdência Complementar para Servidores Públicos,0.198043
928,929,Previdência Regime Geral,0.186167
972,973,Procedimentos Auxiliares na Nova Lei de Licita...,0.185262
16,17,Nova Lei de Licitações e Contratos: aspectos g...,0.180013


#### Construção do perfil do usuário

No componente Content-Based, o perfil de cada usuário é construído a partir do conjunto de cursos presentes em seu histórico. Conceitualmente, os vetores TF-IDF desses cursos podem ser agregados para formar um vetor representativo dos interesses do usuário, que posteriormente é comparado aos cursos candidatos.

Na implementação em lote utilizada nesta POC, essa operação é realizada de forma equivalente por meio da multiplicação da matriz binária de histórico pela matriz de similaridade entre os cursos. Assim, para cada curso candidato, sua pontuação corresponde à soma das similaridades com os cursos anteriormente observados no histórico do usuário.

A utilização da soma, em vez da média das similaridades, não altera a ordenação dos candidatos para um mesmo usuário, pois a divisão pelo número de cursos do histórico corresponderia a um fator constante aplicado a todas as suas pontuações. Essa formulação permite realizar a avaliação de centenas de milhares de usuários de forma mais eficiente.

#### Inspeção das recomendações personalizadas

A aplicação do componente Content-Based a um usuário com seis cursos distintos em seu histórico produziu recomendações coerentes com diferentes temas previamente observados.

O histórico do usuário continha cursos relacionados à gestão e preservação de documentos, previdência dos servidores públicos e à nova Lei de Licitações. Entre as dez recomendações com maiores escores foram identificados novos cursos relacionados a gestão documental, práticas arquivísticas e procedimentos de protocolo, além de conteúdos sobre previdência e licitações.

A inspeção indica que a combinação das representações dos cursos históricos permite construir um perfil capaz de preservar diferentes interesses do usuário, em vez de limitar a recomendação ao tema de apenas uma interação.

Essa análise é qualitativa e tem como objetivo verificar o funcionamento do componente antes de sua avaliação quantitativa no conjunto de validação de 2025.

### 3.7 Função de pontuação Content-Based

A lógica desenvolvida para o usuário de exemplo foi encapsulada em uma função para permitir sua aplicação aos demais usuários.

A função recebe o conjunto de cursos presentes no histórico do usuário, constrói seu perfil pela média das representações TF-IDF e calcula a similaridade desse perfil com os cursos do catálogo. Os itens já presentes no histórico recebem pontuação negativa para impedir sua inclusão nas recomendações.

In [27]:
def calcular_scores_content(ids_cursos_historico):
    indices = catalogo_modelo.index[
        catalogo_modelo["id_curso_canonico"].isin(ids_cursos_historico)
    ].tolist()

    if not indices:
        return None

    perfil = np.asarray(
        matriz_tfidf[indices].mean(axis=0)
    )

    scores = cosine_similarity(
        perfil,
        matriz_tfidf
    ).flatten()

    scores[indices] = -1

    return scores

In [28]:
scores_teste = calcular_scores_content(ids_historico)

indices_teste = scores_teste.argsort()[::-1][:10]

teste_funcao = catalogo_modelo.loc[
    indices_teste,
    ["id_curso_canonico", "nome_curso"]
].copy()

teste_funcao["score_content"] = scores_teste[indices_teste]

teste_funcao

,id_curso_canonico,nome_curso,score_content
492,493,Noções Básicas de Gerenciamento em Serviços Ar...,0.284830
800,801,Elaboração de instrumentos de gestão de docume...,0.267291
546,547,Gestão Documental,0.260675
246,247,"Arquivos Permanentes: recolhimento, processame...",0.253500
769,770,Introdução às Práticas Arquivísticas,0.252638
920,921,Procedimentos de Protocolo,0.241938
210,211,Previdência Complementar para Servidores Públicos,0.198043
928,929,Previdência Regime Geral,0.186167
972,973,Procedimentos Auxiliares na Nova Lei de Licita...,0.185262
16,17,Nova Lei de Licitações e Contratos: aspectos g...,0.180013


## 4. Avaliação do componente Content-Based

Após a construção e a inspeção qualitativa do componente Content-Based, seu desempenho será avaliado utilizando o conjunto temporal de validação correspondente ao ano de 2025.

Para cada usuário avaliável, serão utilizados exclusivamente os cursos presentes em seu histórico anterior a 2025 para a construção do perfil de interesse. As recomendações geradas serão então comparadas aos novos cursos observados para esse usuário durante 2025.

Cursos já presentes no histórico anterior serão excluídos tanto da lista de recomendações quanto da definição dos itens relevantes.

In [29]:
pares_validacao = con.execute(f"""
    WITH historico AS (
        SELECT DISTINCT
            codigo_pessoa,
            id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula < DATE '2025-01-01'
    ),

    interacoes_2025 AS (
        SELECT DISTINCT
            codigo_pessoa,
            id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula >= DATE '2025-01-01'
          AND dt_matricula < DATE '2026-01-01'
    )

    SELECT
        v.codigo_pessoa,
        v.id_curso_canonico
    FROM interacoes_2025 v

    INNER JOIN (
        SELECT DISTINCT codigo_pessoa
        FROM historico
    ) u
        ON v.codigo_pessoa = u.codigo_pessoa

    LEFT JOIN historico h
        ON v.codigo_pessoa = h.codigo_pessoa
       AND v.id_curso_canonico = h.id_curso_canonico

    WHERE h.id_curso_canonico IS NULL
""").df()

print("Pares usuário-curso relevantes:", len(pares_validacao))
print("Usuários avaliáveis:", pares_validacao["codigo_pessoa"].nunique())
print("Cursos relevantes distintos:", pares_validacao["id_curso_canonico"].nunique())

Pares usuário-curso relevantes: 1839939
Usuários avaliáveis: 645919
Cursos relevantes distintos: 975


### 4.2 Histórico disponível para os usuários avaliáveis

Antes da geração das recomendações em escala, foi analisada a quantidade de cursos disponíveis no histórico anterior de cada usuário avaliável.

Essa análise permite caracterizar a quantidade de informação utilizada na construção dos perfis Content-Based e orientar a estratégia computacional empregada para gerar as recomendações.

In [30]:
historico_avaliacao = con.execute(f"""
    WITH usuarios_avaliaveis AS (
        SELECT DISTINCT codigo_pessoa
        FROM pares_validacao
    )

    SELECT DISTINCT
        i.codigo_pessoa,
        i.id_curso_canonico
    FROM read_parquet('{arquivo_interacoes.as_posix()}') i

    INNER JOIN usuarios_avaliaveis u
        ON i.codigo_pessoa = u.codigo_pessoa

    WHERE i.dt_matricula < DATE '2025-01-01'
""").df()

qtd_historico = (
    historico_avaliacao
    .groupby("codigo_pessoa")
    .size()
)

print("Pares usuário-curso no histórico:", len(historico_avaliacao))
print()
print(qtd_historico.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

Pares usuário-curso no histórico: 3316643

count    645919.000000
mean          5.134766
std           8.693111
min           1.000000
25%           1.000000
50%           3.000000
75%           6.000000
90%          12.000000
95%          17.000000
99%          34.000000
max         770.000000
dtype: float64


In [31]:
scores_por_similaridade = (
    similaridade_content[indices_historico]
    .mean(axis=0)
)

scores_por_similaridade[indices_historico] = -1

indices_teste_matriz = scores_por_similaridade.argsort()[::-1][:10]

teste_matriz = catalogo_modelo.loc[
    indices_teste_matriz,
    ["id_curso_canonico", "nome_curso"]
].copy()

teste_matriz["score_medio"] = (
    scores_por_similaridade[indices_teste_matriz]
)

teste_matriz

,id_curso_canonico,nome_curso,score_medio
492,493,Noções Básicas de Gerenciamento em Serviços Ar...,0.129368
800,801,Elaboração de instrumentos de gestão de docume...,0.121401
546,547,Gestão Documental,0.118397
246,247,"Arquivos Permanentes: recolhimento, processame...",0.115138
769,770,Introdução às Práticas Arquivísticas,0.114746
920,921,Procedimentos de Protocolo,0.109886
210,211,Previdência Complementar para Servidores Públicos,0.089949
928,929,Previdência Regime Geral,0.084555
972,973,Procedimentos Auxiliares na Nova Lei de Licita...,0.084145
16,17,Nova Lei de Licitações e Contratos: aspectos g...,0.081761


### 4.3 Preparação da avaliação em escala

Para permitir a geração eficiente das recomendações, foi criado um mapeamento entre o identificador canônico de cada curso e sua posição na matriz de similaridade.

Dessa forma, as relações históricas dos usuários podem ser associadas diretamente às linhas da matriz Content-Based, evitando o recálculo das similaridades textuais para cada usuário.

In [32]:
id_para_indice = {
    id_curso: indice
    for indice, id_curso in enumerate(
        catalogo_modelo["id_curso_canonico"]
    )
}

print("Cursos mapeados:", len(id_para_indice))
print("Exemplo:", list(id_para_indice.items())[:5])

Cursos mapeados: 1030
Exemplo: [(1, 0), (2, 1), (3, 2), (4, 3), (5, 4)]


### 4.4 Organização dos históricos e itens relevantes

Para a avaliação, as relações usuário–curso foram agrupadas por usuário em duas estruturas: o histórico anterior a 2025, utilizado para gerar as recomendações, e os novos cursos observados em 2025, utilizados como itens relevantes.

Essa organização permite comparar diretamente a lista Top-10 produzida pelo recomendador com os cursos efetivamente observados no período de validação.

In [33]:
historicos_por_usuario = (
    historico_avaliacao
    .groupby("codigo_pessoa")["id_curso_canonico"]
    .apply(list)
    .to_dict()
)

alvos_por_usuario = (
    pares_validacao
    .groupby("codigo_pessoa")["id_curso_canonico"]
    .apply(set)
    .to_dict()
)

print("Usuários com histórico:", len(historicos_por_usuario))
print("Usuários com alvos:", len(alvos_por_usuario))

Usuários com histórico: 645919
Usuários com alvos: 645919


### 4.5 Teste do procedimento de avaliação

Antes da execução da avaliação em toda a base de validação, o procedimento foi testado individualmente para verificar a geração das recomendações e a comparação com os itens relevantes observados em 2025.

Para cada usuário, os cursos presentes no histórico são utilizados para calcular a similaridade média com os demais cursos do catálogo. Os cursos já vistos são removidos das possibilidades de recomendação e os dez maiores scores restantes formam a lista Top-10.

In [34]:
usuario_teste = next(iter(alvos_por_usuario))

ids_historico_teste = historicos_por_usuario[usuario_teste]
alvos_teste = alvos_por_usuario[usuario_teste]

indices_historico_teste = [
    id_para_indice[id_curso]
    for id_curso in ids_historico_teste
]

scores_teste = (
    similaridade_content[indices_historico_teste]
    .mean(axis=0)
)

# Remove os cursos que o usuário já havia realizado
scores_teste[indices_historico_teste] = -1

# Seleciona os 10 cursos com maior score
indices_top10 = scores_teste.argsort()[::-1][:10]

ids_top10 = [
    catalogo_modelo.iloc[indice]["id_curso_canonico"]
    for indice in indices_top10
]

print("Usuário:", usuario_teste)
print("Cursos no histórico:", ids_historico_teste)
print("Cursos relevantes em 2025:", alvos_teste)
print("Top-10 recomendado:", ids_top10)

Usuário: 0000014176
Cursos no histórico: [167, 860, 495, 707, 987, 176, 854, 277, 1023, 568, 478]
Cursos relevantes em 2025: {185, 428}
Top-10 recomendado: [np.int64(984), np.int64(634), np.int64(996), np.int64(168), np.int64(995), np.int64(797), np.int64(840), np.int64(245), np.int64(631), np.int64(257)]


### 4.6 Métricas de avaliação

O desempenho das recomendações será avaliado por meio das métricas Recall@10, Hit Rate@10 e NDCG@10.

O **Recall@10** representa a proporção dos itens relevantes do usuário recuperados entre as dez primeiras recomendações. O **Hit Rate@10** indica se pelo menos um item relevante foi encontrado no Top-10. Já o **NDCG@10** considera, além da presença dos itens relevantes, suas posições no ranking, atribuindo maior importância aos acertos localizados nas primeiras posições.

In [35]:
import numpy as np

acertos_teste = [
    id_curso in alvos_teste
    for id_curso in ids_top10
]

numero_acertos = sum(acertos_teste)

recall_teste = numero_acertos / len(alvos_teste)

hit_rate_teste = int(numero_acertos > 0)

dcg_teste = sum(
    1 / np.log2(posicao + 2)
    for posicao, acertou in enumerate(acertos_teste)
    if acertou
)

numero_relevantes_ideal = min(
    len(alvos_teste),
    10
)

idcg_teste = sum(
    1 / np.log2(posicao + 2)
    for posicao in range(numero_relevantes_ideal)
)

ndcg_teste = (
    dcg_teste / idcg_teste
    if idcg_teste > 0
    else 0
)

print("Acertos no Top-10:", numero_acertos)
print("Recall@10:", recall_teste)
print("Hit Rate@10:", hit_rate_teste)
print("NDCG@10:", ndcg_teste)

Acertos no Top-10: 0
Recall@10: 0.0
Hit Rate@10: 0
NDCG@10: 0.0


### 4.7 Avaliação no conjunto de validação

Após a validação individual do procedimento, o componente Content-Based foi avaliado sobre todos os usuários elegíveis do conjunto de validação de 2025.

Para reduzir o consumo de memória, os usuários foram processados em lotes. Para cada usuário, o score de um curso candidato corresponde à similaridade média desse curso em relação aos cursos presentes no histórico anterior a 2025. Os cursos já observados no histórico são excluídos das recomendações.

A lista Top-10 resultante é comparada aos novos cursos observados em 2025 para o cálculo de Recall@10, Hit Rate@10 e NDCG@10.

In [41]:
def avaliar_usuario_content(codigo_pessoa, k=10):
    ids_historico = historicos_por_usuario[codigo_pessoa]
    alvos = alvos_por_usuario[codigo_pessoa]

    indices_historico = [
        id_para_indice[id_curso]
        for id_curso in ids_historico
    ]

    scores = (
        similaridade_content[indices_historico]
        .mean(axis=0)
        .copy()
    )

    # Remove cursos já presentes no histórico
    scores[indices_historico] = -1

    # Localiza os K maiores scores sem ordenar todos os cursos
    indices_top_k = np.argpartition(scores, -k)[-k:]

    # Ordena apenas os K cursos selecionados
    indices_top_k = indices_top_k[
        np.argsort(scores[indices_top_k])[::-1]
    ]

    ids_top_k = [
        int(catalogo_modelo.iloc[indice]["id_curso_canonico"])
        for indice in indices_top_k
    ]

    # Identifica os acertos
    acertos = [
        id_curso in alvos
        for id_curso in ids_top_k
    ]

    numero_acertos = sum(acertos)

    # Recall@K
    recall = numero_acertos / len(alvos)

    # Hit Rate@K
    hit_rate = int(numero_acertos > 0)

    # DCG@K
    dcg = sum(
        1 / np.log2(posicao + 2)
        for posicao, acertou in enumerate(acertos)
        if acertou
    )

    # DCG ideal
    numero_relevantes_ideal = min(len(alvos), k)

    idcg = sum(
        1 / np.log2(posicao + 2)
        for posicao in range(numero_relevantes_ideal)
    )

    ndcg = dcg / idcg if idcg > 0 else 0

    return recall, hit_rate, ndcg

In [42]:
avaliar_usuario_content(usuario_teste)

(0.0, 0, np.float64(0.0))

In [38]:
import time

usuarios_teste_desempenho = list(alvos_por_usuario.keys())[:1000]

inicio = time.time()

resultados_teste_desempenho = [
    avaliar_usuario_content(usuario)
    for usuario in usuarios_teste_desempenho
]

fim = time.time()

tempo_1000 = fim - inicio
tempo_estimado_total = (
    tempo_1000
    * len(alvos_por_usuario)
    / len(usuarios_teste_desempenho)
)

print(f"Tempo para 1.000 usuários: {tempo_1000:.2f} segundos")
print(
    f"Estimativa para {len(alvos_por_usuario):,} usuários: "
    f"{tempo_estimado_total / 60:.2f} minutos"
)

Tempo para 1.000 usuários: 1.77 segundos
Estimativa para 645,919 usuários: 19.05 minutos


In [43]:
inicio = time.time()

resultados_teste_desempenho = [
    avaliar_usuario_content(usuario)
    for usuario in usuarios_teste_desempenho
]

fim = time.time()

tempo_1000_otimizado = fim - inicio

tempo_estimado_total_otimizado = (
    tempo_1000_otimizado
    * len(alvos_por_usuario)
    / len(usuarios_teste_desempenho)
)

print(
    f"Tempo para 1.000 usuários: "
    f"{tempo_1000_otimizado:.2f} segundos"
)

print(
    f"Estimativa para {len(alvos_por_usuario):,} usuários: "
    f"{tempo_estimado_total_otimizado / 60:.2f} minutos"
)

Tempo para 1.000 usuários: 1.63 segundos
Estimativa para 645,919 usuários: 17.57 minutos


### 4.8 Estratégia de processamento em lotes

Embora a avaliação individual reproduza corretamente a lógica do recomendador, sua aplicação sequencial a centenas de milhares de usuários apresenta custo computacional elevado.

Por esse motivo, a avaliação completa será realizada de forma vetorizada e em lotes, permitindo processar simultaneamente múltiplos usuários sem a necessidade de armazenar uma matriz de scores para toda a população de validação.

In [44]:
import scipy

print("Versão do SciPy:", scipy.__version__)

Versão do SciPy: 1.17.1


In [45]:
usuarios_avaliacao = list(alvos_por_usuario.keys())

usuario_para_indice = {
    codigo_pessoa: indice
    for indice, codigo_pessoa in enumerate(usuarios_avaliacao)
}

print("Usuários mapeados:", len(usuario_para_indice))
print("Exemplo:", list(usuario_para_indice.items())[:5])

Usuários mapeados: 645919
Exemplo: [('0000014176', 0), ('000007315b', 1), ('00000c3b76', 2), ('00001fa5b4', 3), ('000027f1b0', 4)]


In [46]:
from scipy.sparse import csr_matrix

linhas = historico_avaliacao["codigo_pessoa"].map(
    usuario_para_indice
).to_numpy()

colunas = historico_avaliacao["id_curso_canonico"].map(
    id_para_indice
).to_numpy()

valores = np.ones(
    len(historico_avaliacao),
    dtype=np.float32
)

matriz_historico = csr_matrix(
    (valores, (linhas, colunas)),
    shape=(
        len(usuario_para_indice),
        len(id_para_indice)
    )
)

print("Formato da matriz:", matriz_historico.shape)
print("Valores armazenados:", matriz_historico.nnz)

Formato da matriz: (645919, 1030)
Valores armazenados: 3316643


In [47]:
indice_usuario_teste = usuario_para_indice[usuario_teste]

indices_cursos_matriz = (
    matriz_historico[indice_usuario_teste]
    .indices
)

ids_cursos_matriz = [
    int(catalogo_modelo.iloc[indice]["id_curso_canonico"])
    for indice in indices_cursos_matriz
]

print("Usuário:", usuario_teste)
print("Histórico original:", sorted(ids_historico_teste))
print("Histórico na matriz:", sorted(ids_cursos_matriz))
print(
    "São iguais:",
    set(ids_historico_teste) == set(ids_cursos_matriz)
)

Usuário: 0000014176
Histórico original: [167, 176, 277, 478, 495, 568, 707, 854, 860, 987, 1023]
Histórico na matriz: [167, 176, 277, 478, 495, 568, 707, 854, 860, 987, 1023]
São iguais: True


In [48]:
linha_usuario = matriz_historico[indice_usuario_teste]

scores_matriz_teste = (
    linha_usuario @ similaridade_content
) / linha_usuario.sum()

scores_matriz_teste = np.asarray(
    scores_matriz_teste
).ravel()

# Remove os cursos já vistos
scores_matriz_teste[indices_cursos_matriz] = -1

# Obtém o Top-10
indices_top10_matriz = np.argpartition(
    scores_matriz_teste,
    -10
)[-10:]

indices_top10_matriz = indices_top10_matriz[
    np.argsort(scores_matriz_teste[indices_top10_matriz])[::-1]
]

ids_top10_matriz = [
    int(catalogo_modelo.iloc[indice]["id_curso_canonico"])
    for indice in indices_top10_matriz
]

print("Top-10 anterior:", ids_top10)
print("Top-10 pela matriz:", ids_top10_matriz)
print("São iguais:", ids_top10 == ids_top10_matriz)

Top-10 anterior: [np.int64(984), np.int64(634), np.int64(996), np.int64(168), np.int64(995), np.int64(797), np.int64(840), np.int64(245), np.int64(631), np.int64(257)]
Top-10 pela matriz: [984, 634, 996, 168, 995, 797, 840, 245, 631, 257]
São iguais: True


In [49]:
inicio = time.time()

# Seleciona as primeiras 1.000 linhas da matriz de histórico
lote_teste = matriz_historico[:1000]

# Soma as similaridades dos cursos presentes no histórico
scores_lote = lote_teste @ similaridade_content

# Calcula a quantidade de cursos no histórico de cada usuário
qtd_cursos_lote = np.asarray(
    lote_teste.sum(axis=1)
).ravel()

# Calcula a média das similaridades
scores_lote = (
    np.asarray(scores_lote)
    / qtd_cursos_lote[:, None]
)

fim = time.time()

print("Formato dos scores:", scores_lote.shape)
print(
    f"Tempo para calcular scores de 1.000 usuários: "
    f"{fim - inicio:.4f} segundos"
)

Formato dos scores: (1000, 1030)
Tempo para calcular scores de 1.000 usuários: 0.0744 segundos


In [50]:
# Cópia para preservar os scores originais do teste
scores_lote_filtrado = scores_lote.copy()

# Localiza as relações usuário-curso existentes no histórico
linhas_vistas, colunas_vistas = lote_teste.nonzero()

# Impede que cursos já realizados sejam recomendados
scores_lote_filtrado[
    linhas_vistas,
    colunas_vistas
] = -1

print(
    "Cursos históricos removidos:",
    len(linhas_vistas)
)

print(
    "Maior score dos cursos já vistos:",
    scores_lote_filtrado[
        linhas_vistas,
        colunas_vistas
    ].max()
)

Cursos históricos removidos: 5168
Maior score dos cursos já vistos: -1.0


In [51]:
# Localiza os 10 maiores scores de cada usuário
indices_top10_lote = np.argpartition(
    scores_lote_filtrado,
    -10,
    axis=1
)[:, -10:]

# Recupera os scores correspondentes
scores_top10_lote = np.take_along_axis(
    scores_lote_filtrado,
    indices_top10_lote,
    axis=1
)

# Ordena apenas os 10 selecionados de cada usuário
ordem_top10 = np.argsort(
    scores_top10_lote,
    axis=1
)[:, ::-1]

indices_top10_lote = np.take_along_axis(
    indices_top10_lote,
    ordem_top10,
    axis=1
)

print("Formato do Top-10:", indices_top10_lote.shape)

print(
    "Top-10 do primeiro usuário:",
    [
        int(catalogo_modelo.iloc[indice]["id_curso_canonico"])
        for indice in indices_top10_lote[0]
    ]
)

Formato do Top-10: (1000, 10)
Top-10 do primeiro usuário: [984, 634, 996, 168, 995, 797, 840, 245, 631, 257]


In [52]:
recalls_teste = []
hits_teste = []
ndcgs_teste = []

for indice_usuario in range(1000):
    codigo_pessoa = usuarios_avaliacao[indice_usuario]
    alvos = alvos_por_usuario[codigo_pessoa]

    ids_recomendados = [
        int(catalogo_modelo.iloc[indice]["id_curso_canonico"])
        for indice in indices_top10_lote[indice_usuario]
    ]

    acertos = [
        id_curso in alvos
        for id_curso in ids_recomendados
    ]

    numero_acertos = sum(acertos)

    # Recall@10
    recall = numero_acertos / len(alvos)

    # Hit Rate@10
    hit = int(numero_acertos > 0)

    # NDCG@10
    dcg = sum(
        1 / np.log2(posicao + 2)
        for posicao, acertou in enumerate(acertos)
        if acertou
    )

    numero_relevantes_ideal = min(len(alvos), 10)

    idcg = sum(
        1 / np.log2(posicao + 2)
        for posicao in range(numero_relevantes_ideal)
    )

    ndcg = dcg / idcg if idcg > 0 else 0

    recalls_teste.append(recall)
    hits_teste.append(hit)
    ndcgs_teste.append(ndcg)


print(f"Recall@10: {np.mean(recalls_teste):.4f}")
print(f"Hit Rate@10: {np.mean(hits_teste):.4f}")
print(f"NDCG@10: {np.mean(ndcgs_teste):.4f}")

Recall@10: 0.1013
Hit Rate@10: 0.1820
NDCG@10: 0.0751


### 4.9 Avaliação completa do Content-Based

Após a validação do procedimento em uma amostra de usuários, a avaliação foi aplicada a todos os usuários elegíveis do conjunto de validação.

O processamento foi realizado em lotes para reduzir o consumo de memória. Em cada lote, os scores dos cursos candidatos foram calculados por meio da multiplicação entre a matriz esparsa de históricos dos usuários e a matriz de similaridade entre os cursos. Os itens já presentes no histórico foram excluídos antes da obtenção das recomendações Top-10.

As métricas individuais foram posteriormente agregadas para obtenção do Recall@10, Hit Rate@10 e NDCG@10 médios do componente Content-Based.

In [53]:
def avaliar_content_em_lotes(tamanho_lote=5000, k=10):
    recalls = []
    hits = []
    ndcgs = []

    total_usuarios = len(usuarios_avaliacao)

    for inicio in range(0, total_usuarios, tamanho_lote):
        fim = min(
            inicio + tamanho_lote,
            total_usuarios
        )

        lote = matriz_historico[inicio:fim]

        # Calcula os scores médios
        scores = lote @ similaridade_content

        qtd_cursos = np.asarray(
            lote.sum(axis=1)
        ).ravel()

        scores = (
            np.asarray(scores)
            / qtd_cursos[:, None]
        )

        # Remove cursos já presentes no histórico
        linhas_vistas, colunas_vistas = lote.nonzero()

        scores[
            linhas_vistas,
            colunas_vistas
        ] = -1

        # Localiza os K maiores scores
        indices_top_k = np.argpartition(
            scores,
            -k,
            axis=1
        )[:, -k:]

        scores_top_k = np.take_along_axis(
            scores,
            indices_top_k,
            axis=1
        )

        # Ordena apenas os K selecionados
        ordem = np.argsort(
            scores_top_k,
            axis=1
        )[:, ::-1]

        indices_top_k = np.take_along_axis(
            indices_top_k,
            ordem,
            axis=1
        )

        # Calcula as métricas de cada usuário
        for linha_local, linha_global in enumerate(
            range(inicio, fim)
        ):
            codigo_pessoa = usuarios_avaliacao[
                linha_global
            ]

            alvos = alvos_por_usuario[
                codigo_pessoa
            ]

            ids_recomendados = [
                int(
                    catalogo_modelo.iloc[indice][
                        "id_curso_canonico"
                    ]
                )
                for indice in indices_top_k[
                    linha_local
                ]
            ]

            acertos = [
                id_curso in alvos
                for id_curso in ids_recomendados
            ]

            numero_acertos = sum(acertos)

            recall = (
                numero_acertos
                / len(alvos)
            )

            hit = int(numero_acertos > 0)

            dcg = sum(
                1 / np.log2(posicao + 2)
                for posicao, acertou
                in enumerate(acertos)
                if acertou
            )

            numero_relevantes_ideal = min(
                len(alvos),
                k
            )

            idcg = sum(
                1 / np.log2(posicao + 2)
                for posicao
                in range(numero_relevantes_ideal)
            )

            ndcg = (
                dcg / idcg
                if idcg > 0
                else 0
            )

            recalls.append(recall)
            hits.append(hit)
            ndcgs.append(ndcg)

        print(
            f"Processados: {fim:,} / "
            f"{total_usuarios:,}"
        )

    return {
        "Recall@10": float(np.mean(recalls)),
        "Hit Rate@10": float(np.mean(hits)),
        "NDCG@10": float(np.mean(ndcgs))
    }

In [54]:
inicio_avaliacao = time.time()

resultado_content = avaliar_content_em_lotes(
    tamanho_lote=5000,
    k=10
)

fim_avaliacao = time.time()

print()
print("Resultado final do Content-Based:")
print(resultado_content)

print(
    f"Tempo total: "
    f"{(fim_avaliacao - inicio_avaliacao) / 60:.2f} minutos"
)

Processados: 5,000 / 645,919
Processados: 10,000 / 645,919
Processados: 15,000 / 645,919
Processados: 20,000 / 645,919
Processados: 25,000 / 645,919
Processados: 30,000 / 645,919
Processados: 35,000 / 645,919
Processados: 40,000 / 645,919
Processados: 45,000 / 645,919
Processados: 50,000 / 645,919
Processados: 55,000 / 645,919
Processados: 60,000 / 645,919
Processados: 65,000 / 645,919
Processados: 70,000 / 645,919
Processados: 75,000 / 645,919
Processados: 80,000 / 645,919
Processados: 85,000 / 645,919
Processados: 90,000 / 645,919
Processados: 95,000 / 645,919
Processados: 100,000 / 645,919
Processados: 105,000 / 645,919
Processados: 110,000 / 645,919
Processados: 115,000 / 645,919
Processados: 120,000 / 645,919
Processados: 125,000 / 645,919
Processados: 130,000 / 645,919
Processados: 135,000 / 645,919
Processados: 140,000 / 645,919
Processados: 145,000 / 645,919
Processados: 150,000 / 645,919
Processados: 155,000 / 645,919
Processados: 160,000 / 645,919
Processados: 165,000 / 645,9

### 4.10 Resultados do Content-Based

A avaliação do componente Content-Based foi realizada sobre 645.919 usuários com histórico anterior a 2025 e pelo menos um novo curso observado durante o período de validação.

O modelo apresentou **Recall@10 de 0,0956**, indicando que, em média, aproximadamente 9,56% dos novos cursos relevantes para cada usuário foram recuperados entre as dez primeiras recomendações. O **Hit Rate@10 foi de 0,1772**, o que significa que aproximadamente 17,72% dos usuários avaliados tiveram pelo menos um de seus novos cursos futuros presente na lista Top-10.

O **NDCG@10 foi de 0,0682**, métrica que considera não apenas a recuperação dos itens relevantes, mas também suas posições no ranking de recomendações.

Esses resultados constituem a linha de base do componente baseado em conteúdo para a etapa de validação. O desempenho deverá ser posteriormente comparado ao componente de Filtragem Colaborativa e ao modelo híbrido, permitindo verificar a contribuição de cada abordagem.

É importante destacar que as métricas refletem a capacidade do modelo de recuperar interações futuras observadas nos dados, não sendo possível assumir que cursos não realizados pelo usuário sejam necessariamente recomendações irrelevantes.

## 5. Filtragem Colaborativa Item-Based

O segundo componente do sistema de recomendação utiliza Filtragem Colaborativa baseada em itens (Item-Based Collaborative Filtering).

Diferentemente do componente Content-Based, que utiliza as características textuais dos cursos, a Filtragem Colaborativa identifica relações entre os cursos a partir do comportamento dos usuários. A premissa é que cursos frequentemente presentes nos históricos dos mesmos usuários podem apresentar uma relação de interesse, mesmo que seus conteúdos textuais não sejam necessariamente semelhantes.

Nesta POC, as interações são tratadas como feedback implícito binário: a presença de um curso no histórico do usuário indica uma interação, independentemente da situação final da matrícula. Relações repetidas entre o mesmo usuário e curso são consolidadas em uma única interação.

A abordagem Item-Based foi escolhida devido à estrutura da base: embora existam milhões de usuários, o catálogo possui apenas 1.030 cursos canônicos, tornando computacionalmente viável calcular relações entre os itens sem construir uma matriz densa usuário × usuário.

### 5.1 Construção da base de treinamento

Para o treinamento do componente colaborativo foram utilizadas as interações ocorridas até 31 de dezembro de 2024, respeitando a divisão temporal definida anteriormente.

As relações repetidas entre um mesmo usuário e curso foram consolidadas, de forma que cada par usuário–curso represente uma interação implícita binária.

In [55]:
pares_treino_cf = con.execute(f"""
    SELECT DISTINCT
        codigo_pessoa,
        id_curso_canonico
    FROM read_parquet('{arquivo_interacoes.as_posix()}')
    WHERE dt_matricula < DATE '2025-01-01'
""").df()

print("Pares usuário-curso:", len(pares_treino_cf))
print(
    "Usuários distintos:",
    pares_treino_cf["codigo_pessoa"].nunique()
)
print(
    "Cursos distintos:",
    pares_treino_cf["id_curso_canonico"].nunique()
)

Pares usuário-curso: 11778710
Usuários distintos: 4270327
Cursos distintos: 788


### 5.2 Representação esparsa das interações

As interações de treinamento foram representadas em uma matriz esparsa usuário × curso, na qual o valor 1 indica a existência de uma interação entre determinado usuário e curso.

A representação esparsa permite trabalhar com milhões de usuários sem a necessidade de armazenar explicitamente as posições sem interação, que constituem a maior parte da matriz.

In [56]:
usuarios_treino_cf = (
    pares_treino_cf["codigo_pessoa"]
    .drop_duplicates()
    .tolist()
)

cursos_treino_cf = (
    pares_treino_cf["id_curso_canonico"]
    .drop_duplicates()
    .tolist()
)

usuario_para_indice_cf = {
    usuario: indice
    for indice, usuario in enumerate(usuarios_treino_cf)
}

curso_para_indice_cf = {
    curso: indice
    for indice, curso in enumerate(cursos_treino_cf)
}

print("Usuários mapeados:", len(usuario_para_indice_cf))
print("Cursos mapeados:", len(curso_para_indice_cf))

Usuários mapeados: 4270327
Cursos mapeados: 788


In [57]:
linhas_cf = pares_treino_cf["codigo_pessoa"].map(
    usuario_para_indice_cf
).to_numpy()

colunas_cf = pares_treino_cf["id_curso_canonico"].map(
    curso_para_indice_cf
).to_numpy()

valores_cf = np.ones(
    len(pares_treino_cf),
    dtype=np.float32
)

matriz_cf = csr_matrix(
    (valores_cf, (linhas_cf, colunas_cf)),
    shape=(
        len(usuario_para_indice_cf),
        len(curso_para_indice_cf)
    )
)

print("Formato da matriz:", matriz_cf.shape)
print("Interações armazenadas:", matriz_cf.nnz)

Formato da matriz: (4270327, 788)
Interações armazenadas: 11778710


### 5.3 Coocorrência entre cursos

A partir da matriz esparsa de interações, foi calculada a coocorrência entre os cursos. Essa medida representa quantos usuários distintos possuem cada par de cursos em seus históricos de treinamento.

A matriz de coocorrência é obtida pelo produto entre a transposta da matriz usuário × curso e a própria matriz de interações. Os valores da diagonal representam a quantidade de usuários que interagiram com cada curso, enquanto os demais valores representam o número de usuários compartilhados entre pares de cursos.

In [58]:
inicio_cf = time.time()

matriz_coocorrencia = (
    matriz_cf.T @ matriz_cf
)

fim_cf = time.time()

print(
    "Formato da matriz de coocorrência:",
    matriz_coocorrencia.shape
)

print(
    "Valores armazenados:",
    matriz_coocorrencia.nnz
)

print(
    f"Tempo de cálculo: "
    f"{fim_cf - inicio_cf:.2f} segundos"
)

Formato da matriz de coocorrência: (788, 788)
Valores armazenados: 608040
Tempo de cálculo: 9.05 segundos


### 5.4 Cálculo da similaridade entre os cursos

A coocorrência absoluta pode favorecer cursos muito populares, pois esses cursos tendem naturalmente a compartilhar usuários com uma quantidade maior de itens.

Para reduzir esse efeito, foi utilizada a similaridade de cosseno entre os vetores de interação dos cursos. Essa medida considera tanto a quantidade de usuários compartilhados quanto o número total de usuários de cada curso, produzindo valores normalizados entre 0 e 1.

A diagonal da matriz de similaridade foi posteriormente zerada, uma vez que a similaridade de um curso consigo mesmo não é relevante para a geração das recomendações.

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

inicio_cf = time.time()

similaridade_cf = cosine_similarity(
    matriz_cf.T,
    dense_output=True
)

np.fill_diagonal(similaridade_cf, 0)

fim_cf = time.time()

print("Formato:", similaridade_cf.shape)
print("Similaridade mínima:", similaridade_cf.min())
print("Similaridade máxima:", similaridade_cf.max())
print(f"Tempo de cálculo: {fim_cf - inicio_cf:.2f} segundos")

Formato: (788, 788)
Similaridade mínima: 0.0
Similaridade máxima: 0.8705716
Tempo de cálculo: 7.50 segundos


### 5.5 Inspeção qualitativa das similaridades colaborativas

Antes da avaliação quantitativa do modelo, foram inspecionados alguns dos pares de cursos com maior similaridade colaborativa.

Essa análise permite verificar quais relações foram identificadas exclusivamente a partir do comportamento dos usuários. Diferentemente do Content-Based, cursos colaborativamente semelhantes não precisam apresentar necessariamente conteúdos textuais semelhantes, pois a relação é determinada pela presença conjunta nos históricos dos usuários.

In [60]:
curso_exemplo_cf = pares_treino_cf[
    pares_treino_cf["id_curso_canonico"] == 34
]["id_curso_canonico"].unique()

print("Curso encontrado no treinamento:", curso_exemplo_cf)

Curso encontrado no treinamento: [34]


In [61]:
id_curso_exemplo = 34

indice_curso_cf = curso_para_indice_cf[id_curso_exemplo]

scores_curso_cf = similaridade_cf[indice_curso_cf]

indices_similares_cf = np.argpartition(
    scores_curso_cf, -10
)[-10:]

indices_similares_cf = indices_similares_cf[
    np.argsort(scores_curso_cf[indices_similares_cf])[::-1]
]

indice_para_curso_cf = np.array(cursos_treino_cf)

ids_similares_cf = indice_para_curso_cf[
    indices_similares_cf
]

resultado_similares_cf = catalogo_modelo[
    catalogo_modelo["id_curso_canonico"].isin(ids_similares_cf)
][
    ["id_curso_canonico", "nome_curso"]
].copy()

score_por_curso = {
    int(indice_para_curso_cf[indice]): float(scores_curso_cf[indice])
    for indice in indices_similares_cf
}

resultado_similares_cf["similaridade_cf"] = (
    resultado_similares_cf["id_curso_canonico"]
    .map(score_por_curso)
)

resultado_similares_cf = resultado_similares_cf.sort_values(
    "similaridade_cf",
    ascending=False
)

resultado_similares_cf

,id_curso_canonico,nome_curso,similaridade_cf
729,730,Introdução à Ciência de Dados - Redes Complexas,0.248811
136,137,Governo Data-Driven: transformação orientada p...,0.113703
153,154,Introdução à Ciência de Dados - Modelos de Agr...,0.113134
837,838,Introdução à Ciência de Dados - Modelos de Reg...,0.106909
263,264,Introdução à Ciência de Dados - Estatística Es...,0.104098
577,578,Introdução à Ciência de Dados - Descoberta de ...,0.096487
981,982,Introdução à Ciência de Dados - Conceitos e Fe...,0.087616
382,383,Introdução à Ciência de Dados - Conjuntos Freq...,0.082671
183,184,Segurança da Informação para Todos,0.077773
627,628,Introdução à Ciência de Dados - Modelos de Cla...,0.077526


A inspeção qualitativa das similaridades colaborativas apresentou relações coerentes com o curso analisado. Entre os dez cursos com maior similaridade, oito pertencem à mesma sequência de cursos de Introdução à Ciência de Dados, indicando que usuários que interagem com um dos cursos dessa sequência frequentemente apresentam interesses compartilhados com os demais.

Também foram identificados cursos que não pertencem diretamente à mesma sequência, como "Governo Data-Driven" e "Segurança da Informação para Todos". Essas relações ilustram uma característica importante da Filtragem Colaborativa: a similaridade é derivada dos padrões de interação dos usuários e, portanto, pode identificar associações que não dependem exclusivamente da proximidade textual entre os conteúdos.

A comparação qualitativa com o componente Content-Based mostra que as duas abordagens capturam informações parcialmente complementares, reforçando a proposta de combiná-las posteriormente no sistema híbrido.

### 5.6 Alinhamento com o catálogo completo

O componente colaborativo foi treinado com 788 cursos que possuíam histórico de interação até o final de 2024, enquanto o catálogo canônico contém 1.030 cursos.

Para permitir a integração posterior com o componente Content-Based, a matriz colaborativa foi alinhada ao espaço completo do catálogo. Os cursos sem histórico no período de treinamento permanecem sem similaridade colaborativa, representada por valor zero.

Essa característica evidencia o problema de cold-start de itens na Filtragem Colaborativa e constitui uma das principais motivações para sua combinação com o Content-Based.

In [62]:
similaridade_cf_completa = np.zeros(
    (len(catalogo_modelo), len(catalogo_modelo)),
    dtype=np.float32
)

indices_catalogo_cf = np.array([
    id_para_indice[id_curso]
    for id_curso in cursos_treino_cf
])

similaridade_cf_completa[
    np.ix_(indices_catalogo_cf, indices_catalogo_cf)
] = similaridade_cf

print("Formato:", similaridade_cf_completa.shape)

print(
    "Cursos com histórico colaborativo:",
    len(indices_catalogo_cf)
)

print(
    "Cursos sem histórico colaborativo:",
    len(catalogo_modelo) - len(indices_catalogo_cf)
)

Formato: (1030, 1030)
Cursos com histórico colaborativo: 788
Cursos sem histórico colaborativo: 242


### 5.7 Preparação da avaliação do componente colaborativo

O componente colaborativo é avaliado utilizando exatamente os mesmos usuários, cursos relevantes e métricas empregados na avaliação do Content-Based, garantindo comparabilidade entre as abordagens.

Para tornar o processamento mais eficiente, os cursos relevantes observados em 2025 foram representados em uma matriz esparsa usuário × curso. Dessa forma, os acertos presentes nas recomendações Top-10 podem ser identificados de forma vetorizada durante a avaliação.

In [63]:
linhas_alvo = (
    pares_validacao["codigo_pessoa"]
    .map(usuario_para_indice)
    .to_numpy()
)

colunas_alvo = (
    pares_validacao["id_curso_canonico"]
    .map(id_para_indice)
    .to_numpy()
)

valores_alvo = np.ones(
    len(pares_validacao),
    dtype=np.float32
)

matriz_alvos = csr_matrix(
    (valores_alvo, (linhas_alvo, colunas_alvo)),
    shape=(
        len(usuarios_avaliacao),
        len(catalogo_modelo)
    )
)

print("Formato:", matriz_alvos.shape)
print("Alvos armazenados:", matriz_alvos.nnz)

Formato: (645919, 1030)
Alvos armazenados: 1839939


### 5.8 Avaliação do Item-Based Collaborative Filtering

A avaliação do componente colaborativo utiliza Recall@10, Hit Rate@10 e NDCG@10, mantendo o mesmo protocolo empregado no Content-Based.

Para cada usuário, os scores dos cursos candidatos são calculados a partir da similaridade colaborativa dos cursos presentes em seu histórico. Cursos já presentes no histórico são removidos das recomendações.

Como o componente colaborativo não possui informações sobre os 242 cursos sem histórico até 2024, esses itens não recebem evidência colaborativa. Os alvos correspondentes a esses cursos permanecem, entretanto, na avaliação geral, permitindo que a limitação de cold-start do método seja refletida em seu desempenho.

In [65]:
def avaliar_similaridade_em_lotes(
    matriz_similaridade,
    tamanho_lote=5000,
    k=10,
    limite_usuarios=None,
    mascara_candidatos=None
):
    total_usuarios = len(usuarios_avaliacao)

    if limite_usuarios is not None:
        total_usuarios = min(
            total_usuarios,
            limite_usuarios
        )

    soma_recall = 0.0
    soma_hit = 0.0
    soma_ndcg = 0.0

    descontos = 1 / np.log2(
        np.arange(k) + 2
    )

    descontos_acumulados = np.cumsum(descontos)

    for inicio in range(0, total_usuarios, tamanho_lote):
        fim = min(
            inicio + tamanho_lote,
            total_usuarios
        )

        lote_historico = matriz_historico[inicio:fim]

        scores = (
            lote_historico @ matriz_similaridade
        )

        scores = np.asarray(scores)

        # Remove cursos sem evidência colaborativa, quando necessário
        if mascara_candidatos is not None:
            scores[:, ~mascara_candidatos] = -np.inf

        # Remove cursos já vistos pelo usuário
        linhas_vistas, colunas_vistas = (
            lote_historico.nonzero()
        )

        scores[
            linhas_vistas,
            colunas_vistas
        ] = -np.inf

        # Seleciona o Top-K
        indices_top_k = np.argpartition(
            scores,
            -k,
            axis=1
        )[:, -k:]

        scores_top_k = np.take_along_axis(
            scores,
            indices_top_k,
            axis=1
        )

        ordem = np.argsort(
            scores_top_k,
            axis=1
        )[:, ::-1]

        indices_top_k = np.take_along_axis(
            indices_top_k,
            ordem,
            axis=1
        )

        # Busca os alvos reais do mesmo lote
        alvos_lote = (
            matriz_alvos[inicio:fim]
            .toarray()
            .astype(bool)
        )

        acertos = np.take_along_axis(
            alvos_lote,
            indices_top_k,
            axis=1
        )

        numero_acertos = acertos.sum(axis=1)

        quantidade_alvos = (
            alvos_lote.sum(axis=1)
        )

        recall = (
            numero_acertos
            / quantidade_alvos
        )

        hit = (
            numero_acertos > 0
        )

        dcg = (
            acertos * descontos
        ).sum(axis=1)

        quantidade_ideal = np.minimum(
            quantidade_alvos,
            k
        ).astype(int)

        idcg = descontos_acumulados[
            quantidade_ideal - 1
        ]

        ndcg = dcg / idcg

        soma_recall += recall.sum()
        soma_hit += hit.sum()
        soma_ndcg += ndcg.sum()

    return {
        "Recall@10": float(
            soma_recall / total_usuarios
        ),
        "Hit Rate@10": float(
            soma_hit / total_usuarios
        ),
        "NDCG@10": float(
            soma_ndcg / total_usuarios
        )
    }

In [66]:
mascara_cursos_cf = np.zeros(
    len(catalogo_modelo),
    dtype=bool
)

mascara_cursos_cf[
    indices_catalogo_cf
] = True

resultado_teste_cf = avaliar_similaridade_em_lotes(
    matriz_similaridade=similaridade_cf_completa,
    tamanho_lote=1000,
    k=10,
    limite_usuarios=1000,
    mascara_candidatos=mascara_cursos_cf
)

resultado_teste_cf

{'Recall@10': 0.11431465900501436,
 'Hit Rate@10': 0.214,
 'NDCG@10': 0.07482590705386516}

A avaliação completa foi realizada sobre os 645.919 usuários elegíveis da validação temporal de 2025. Para cada usuário, foram geradas dez recomendações a partir das similaridades colaborativas dos cursos presentes em seu histórico até 2024.

Os cursos sem histórico no período de treinamento foram mantidos entre os alvos relevantes da avaliação, mas não foram considerados candidatos pontuáveis pelo componente colaborativo, uma vez que o método não possui evidência comportamental para estimar suas relações com outros itens. Dessa forma, a avaliação também incorpora o impacto do cold-start de itens sobre o desempenho do modelo.

In [67]:
inicio_avaliacao_cf = time.time()

resultado_cf = avaliar_similaridade_em_lotes(
    matriz_similaridade=similaridade_cf_completa,
    tamanho_lote=5000,
    k=10,
    mascara_candidatos=mascara_cursos_cf
)

fim_avaliacao_cf = time.time()

print(resultado_cf)

print(
    f"Tempo total: "
    f"{(fim_avaliacao_cf - inicio_avaliacao_cf) / 60:.2f} minutos"
)

{'Recall@10': 0.10975053212221846, 'Hit Rate@10': 0.20021395871618577, 'NDCG@10': 0.07151259159632807}
Tempo total: 0.26 minutos


### 5.9 Resultados do Item-Based Collaborative Filtering

A avaliação do Item-Based Collaborative Filtering foi realizada sobre os mesmos 645.919 usuários utilizados na avaliação do Content-Based, preservando o mesmo conjunto de alvos futuros e o mesmo protocolo de avaliação.

Os resultados obtidos foram:

- **Recall@10:** 0,1098
- **Hit Rate@10:** 0,2002
- **NDCG@10:** 0,0715

O Recall@10 indica que aproximadamente 10,98% dos novos cursos relevantes observados em 2025 foram recuperados entre as dez primeiras recomendações. O Hit Rate@10 mostra que aproximadamente 20,02% dos usuários receberam pelo menos uma recomendação correspondente a um curso com o qual interagiram posteriormente. O NDCG@10 de 0,0715 considera também a posição dos itens relevantes no ranking.

Sob o mesmo protocolo de validação, o componente colaborativo apresentou valores superiores aos obtidos pelo Content-Based nas três métricas. Entretanto, a Filtragem Colaborativa permanece limitada aos cursos que possuem histórico no período de treinamento. Os 242 cursos sem histórico até 2024 não podem receber pontuação colaborativa, caracterizando um problema de cold-start de itens.

Essa limitação, combinada à capacidade do Content-Based de representar cursos a partir de seus próprios atributos, reforça a motivação para a construção do sistema híbrido.

## 6. Sistema Híbrido de Recomendação

O sistema híbrido combina as pontuações produzidas pelos componentes Content-Based e Item-Based Collaborative Filtering.

A combinação busca aproveitar características complementares das duas abordagens. O Content-Based utiliza os atributos dos próprios cursos e consegue produzir relações para itens sem histórico comportamental, enquanto o componente colaborativo explora padrões de interesse observados nos históricos dos usuários.

A pontuação híbrida é definida pela combinação ponderada dos dois componentes:

\[
Score_{híbrido} = \alpha \cdot Score_{content} + (1-\alpha) \cdot Score_{colaborativo}
\]

em que \(\alpha\) controla a contribuição do Content-Based. Valores maiores de \(\alpha\) aumentam a influência do conteúdo, enquanto valores menores aumentam a influência do componente colaborativo.

Os pesos são avaliados utilizando o conjunto de validação temporal de 2025, evitando a escolha arbitrária da configuração do modelo.

### 6.1 Normalização dos componentes

Como as pontuações produzidas pelo Content-Based e pelo Item-Based Collaborative Filtering possuem escalas diferentes, sua combinação direta poderia favorecer um dos componentes apenas pela magnitude numérica dos scores.

Antes da combinação, as pontuações de cada componente são normalizadas individualmente para cada usuário, dividindo-se os valores pelo maior score válido produzido pelo respectivo modelo. Essa transformação preserva a ordenação dos candidatos dentro de cada componente e permite que o parâmetro de ponderação represente de forma mais adequada a contribuição relativa das duas abordagens.

In [68]:
lote_teste_hibrido = matriz_historico[:1]

scores_content_teste = np.asarray(
    lote_teste_hibrido @ similaridade_content
)

scores_cf_teste = np.asarray(
    lote_teste_hibrido @ similaridade_cf_completa
)

linhas_vistas, colunas_vistas = lote_teste_hibrido.nonzero()

scores_content_teste[
    linhas_vistas,
    colunas_vistas
] = 0

scores_cf_teste[
    linhas_vistas,
    colunas_vistas
] = 0

print(
    "Maior score Content-Based:",
    scores_content_teste.max()
)

print(
    "Maior score Collaborative:",
    scores_cf_teste.max()
)

Maior score Content-Based: 1.8479886626090753
Maior score Collaborative: 1.369168


In [69]:
max_content = scores_content_teste.max(axis=1, keepdims=True)
max_cf = scores_cf_teste.max(axis=1, keepdims=True)

scores_content_normalizados = (
    scores_content_teste
    / np.where(max_content > 0, max_content, 1)
)

scores_cf_normalizados = (
    scores_cf_teste
    / np.where(max_cf > 0, max_cf, 1)
)

print(
    "Maior score Content-Based normalizado:",
    scores_content_normalizados.max()
)

print(
    "Maior score Collaborative normalizado:",
    scores_cf_normalizados.max()
)

Maior score Content-Based normalizado: 1.0
Maior score Collaborative normalizado: 1.0


### 6.2 Avaliação dos pesos de combinação

Para determinar a contribuição relativa dos componentes Content-Based e colaborativo, foram avaliadas diferentes configurações do parâmetro α no conjunto de validação de 2025.

Foram inicialmente considerados os valores 0,00, 0,25, 0,50, 0,75 e 1,00. Dessa forma, os extremos representam os componentes individuais e os valores intermediários representam diferentes combinações entre conteúdo e comportamento.

Para cada usuário, os scores dos dois componentes são normalizados antes da combinação. Os cursos já presentes no histórico do usuário são excluídos dos candidatos, enquanto os cursos sem histórico colaborativo permanecem elegíveis por meio de sua pontuação Content-Based.

In [70]:
def avaliar_hibrido(
    alphas,
    tamanho_lote=5000,
    k=10
):
    resultados = {
        alpha: {
            "recall": 0.0,
            "hit": 0.0,
            "ndcg": 0.0
        }
        for alpha in alphas
    }

    total_usuarios = len(usuarios_avaliacao)

    descontos = 1 / np.log2(
        np.arange(k) + 2
    )

    descontos_acumulados = np.cumsum(
        descontos
    )

    for inicio in range(
        0,
        total_usuarios,
        tamanho_lote
    ):
        fim = min(
            inicio + tamanho_lote,
            total_usuarios
        )

        lote_historico = matriz_historico[
            inicio:fim
        ]

        # Scores dos dois componentes
        scores_content = np.asarray(
            lote_historico
            @ similaridade_content
        )

        scores_cf = np.asarray(
            lote_historico
            @ similaridade_cf_completa
        )

        # Cursos já vistos
        linhas_vistas, colunas_vistas = (
            lote_historico.nonzero()
        )

        scores_content[
            linhas_vistas,
            colunas_vistas
        ] = 0

        scores_cf[
            linhas_vistas,
            colunas_vistas
        ] = 0

        # Normalização por usuário
        max_content = scores_content.max(
            axis=1,
            keepdims=True
        )

        max_cf = scores_cf.max(
            axis=1,
            keepdims=True
        )

        scores_content = (
            scores_content
            / np.where(
                max_content > 0,
                max_content,
                1
            )
        )

        scores_cf = (
            scores_cf
            / np.where(
                max_cf > 0,
                max_cf,
                1
            )
        )

        # Alvos reais
        alvos_lote = (
            matriz_alvos[inicio:fim]
            .toarray()
            .astype(bool)
        )

        quantidade_alvos = (
            alvos_lote.sum(axis=1)
        )

        quantidade_ideal = np.minimum(
            quantidade_alvos,
            k
        ).astype(int)

        idcg = descontos_acumulados[
            quantidade_ideal - 1
        ]

        # Teste de cada alpha
        for alpha in alphas:

            scores_hibridos = (
                alpha * scores_content
                + (1 - alpha) * scores_cf
            )

            # Remove definitivamente os vistos
            scores_hibridos[
                linhas_vistas,
                colunas_vistas
            ] = -np.inf

            indices_top_k = np.argpartition(
                scores_hibridos,
                -k,
                axis=1
            )[:, -k:]

            scores_top_k = np.take_along_axis(
                scores_hibridos,
                indices_top_k,
                axis=1
            )

            ordem = np.argsort(
                scores_top_k,
                axis=1
            )[:, ::-1]

            indices_top_k = np.take_along_axis(
                indices_top_k,
                ordem,
                axis=1
            )

            acertos = np.take_along_axis(
                alvos_lote,
                indices_top_k,
                axis=1
            )

            numero_acertos = acertos.sum(
                axis=1
            )

            recall = (
                numero_acertos
                / quantidade_alvos
            )

            hit = numero_acertos > 0

            dcg = (
                acertos * descontos
            ).sum(axis=1)

            ndcg = dcg / idcg

            resultados[alpha]["recall"] += (
                recall.sum()
            )

            resultados[alpha]["hit"] += (
                hit.sum()
            )

            resultados[alpha]["ndcg"] += (
                ndcg.sum()
            )

    linhas_resultado = []

    for alpha in alphas:
        linhas_resultado.append({
            "alpha": alpha,
            "peso_content": alpha,
            "peso_cf": 1 - alpha,
            "Recall@10":
                resultados[alpha]["recall"]
                / total_usuarios,
            "Hit Rate@10":
                resultados[alpha]["hit"]
                / total_usuarios,
            "NDCG@10":
                resultados[alpha]["ndcg"]
                / total_usuarios
        })

    return pd.DataFrame(linhas_resultado)

In [71]:
alphas_iniciais = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00
]

inicio_hibrido = time.time()

resultados_hibrido = avaliar_hibrido(
    alphas=alphas_iniciais,
    tamanho_lote=5000,
    k=10
)

fim_hibrido = time.time()

display(resultados_hibrido)

print(
    f"Tempo total: "
    f"{(fim_hibrido - inicio_hibrido) / 60:.2f} minutos"
)

,alpha,peso_content,peso_cf,Recall@10,Hit Rate@10,NDCG@10
0,0.00,0.00,1.00,0.109751,0.200214,0.071513
1,0.25,0.25,0.75,0.117682,0.209249,0.076061
2,0.50,0.50,0.50,0.116086,0.206069,0.077774
3,0.75,0.75,0.25,0.108240,0.194766,0.075377
4,1.00,1.00,0.00,0.095599,0.177182,0.068164


Tempo total: 2.05 minutos


### 6.3 Refinamento do parâmetro α

A avaliação inicial mostrou que as combinações intermediárias apresentaram resultados superiores aos componentes isolados. Os melhores resultados concentraram-se entre α = 0,25 e α = 0,50.

Por esse motivo, foi realizada uma segunda avaliação com valores intermediários nessa região, permitindo observar com maior precisão o comportamento das métricas e selecionar a configuração a ser utilizada posteriormente no teste temporal.

In [72]:
alphas_refinados = [
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

inicio_refinamento = time.time()

resultados_refinados = avaliar_hibrido(
    alphas=alphas_refinados,
    tamanho_lote=5000,
    k=10
)

fim_refinamento = time.time()

display(resultados_refinados)

print(
    f"Tempo total: "
    f"{(fim_refinamento - inicio_refinamento) / 60:.2f} minutos"
)

,alpha,peso_content,peso_cf,Recall@10,Hit Rate@10,NDCG@10
0,0.25,0.25,0.75,0.117682,0.209249,0.076061
1,0.30,0.30,0.70,0.114157,0.203273,0.075905
2,0.35,0.35,0.65,0.114857,0.204411,0.076381
3,0.40,0.40,0.60,0.115863,0.205699,0.076941
4,0.45,0.45,0.55,0.116231,0.206294,0.077360
5,0.50,0.50,0.50,0.116086,0.206069,0.077774


Tempo total: 3.93 minutos


Como o NDCG@10 continuou aumentando até o limite superior da faixa inicialmente refinada, foi realizada uma última busca entre α = 0,50 e α = 0,75. O NDCG@10 foi adotado como métrica principal para a seleção do parâmetro, pois considera simultaneamente a recuperação dos itens relevantes e suas posições no ranking. Recall@10 e Hit Rate@10 foram mantidos como métricas complementares.

In [73]:
alphas_finais = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75
]

inicio_refinamento_final = time.time()

resultados_finais_alpha = avaliar_hibrido(
    alphas=alphas_finais,
    tamanho_lote=5000,
    k=10
)

fim_refinamento_final = time.time()

display(resultados_finais_alpha)

print(
    f"Tempo total: "
    f"{(fim_refinamento_final - inicio_refinamento_final) / 60:.2f} minutos"
)

,alpha,peso_content,peso_cf,Recall@10,Hit Rate@10,NDCG@10
0,0.50,0.50,0.50,0.116086,0.206069,0.077774
1,0.55,0.55,0.45,0.115341,0.204933,0.079077
2,0.60,0.60,0.40,0.114644,0.203999,0.078588
3,0.65,0.65,0.35,0.112899,0.201510,0.077814
4,0.70,0.70,0.30,0.110371,0.198088,0.076549
5,0.75,0.75,0.25,0.108240,0.194766,0.075377


Tempo total: 4.65 minutos


### 6.4 Seleção da configuração híbrida

O refinamento final identificou o maior NDCG@10 em α = 0,55, correspondente à combinação de 55% da pontuação normalizada do Content-Based e 45% da pontuação normalizada do Item-Based Collaborative Filtering.

Nessa configuração, foram obtidos na validação de 2025:

- **Recall@10:** 0,1153
- **Hit Rate@10:** 0,2049
- **NDCG@10:** 0,0791

Embora outras configurações tenham apresentado valores superiores de Recall@10 ou Hit Rate@10, o parâmetro α foi selecionado com base no NDCG@10, definido como métrica principal por considerar não apenas a recuperação dos itens relevantes, mas também suas posições no ranking de recomendações.

A configuração α = 0,55 foi, portanto, fixada após a validação temporal de 2025. O conjunto de teste de 2026 não é utilizado para novos ajustes de hiperparâmetros, sendo reservado para a avaliação final da configuração selecionada.

## 7. Avaliação temporal final

Após a seleção da configuração híbrida no conjunto de validação de 2025, o modelo é submetido à avaliação temporal final utilizando as interações observadas em 2026.

Para essa etapa, o componente colaborativo é reconstruído utilizando todas as interações disponíveis até 31 de dezembro de 2025. O parâmetro α permanece fixado em 0,55, conforme definido exclusivamente durante a validação.

As interações de 2026, disponíveis até 2 de setembro, são utilizadas somente como conjunto de teste e não participam do treinamento ou da seleção dos hiperparâmetros.

In [74]:
resumo_treino_final = con.execute(f"""
    SELECT
        COUNT(*) AS pares_usuario_curso,
        COUNT(DISTINCT codigo_pessoa) AS usuarios,
        COUNT(DISTINCT id_curso_canonico) AS cursos
    FROM (
        SELECT DISTINCT
            codigo_pessoa,
            id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula < DATE '2026-01-01'
    )
""").df()

resumo_treino_final

,pares_usuario_curso,usuarios,cursos
0,16016460,5504668,975


### 7.1 Reconstrução do componente colaborativo

Após a definição dos hiperparâmetros na validação de 2025, o componente colaborativo foi reconstruído utilizando todas as interações históricas disponíveis até 31 de dezembro de 2025.

A incorporação do período de validação ampliou a cobertura comportamental de 788 para 975 cursos canônicos. Dessa forma, apenas 55 dos 1.030 cursos do catálogo permanecem sem histórico colaborativo antes do período de teste de 2026.

As relações repetidas entre usuário e curso continuam sendo consolidadas em uma única interação implícita binária.

In [75]:
pares_treino_final = con.execute(f"""
    SELECT DISTINCT
        codigo_pessoa,
        id_curso_canonico
    FROM read_parquet('{arquivo_interacoes.as_posix()}')
    WHERE dt_matricula < DATE '2026-01-01'
""").df()

print("Pares:", len(pares_treino_final))
print(
    "Usuários:",
    pares_treino_final["codigo_pessoa"].nunique()
)
print(
    "Cursos:",
    pares_treino_final["id_curso_canonico"].nunique()
)

Pares: 16016460
Usuários: 5504668
Cursos: 975


In [76]:
indices_usuarios_final, usuarios_treino_final = pd.factorize(
    pares_treino_final["codigo_pessoa"],
    sort=False
)

colunas_final = (
    pares_treino_final["id_curso_canonico"]
    .map(id_para_indice)
    .to_numpy()
)

valores_final = np.ones(
    len(pares_treino_final),
    dtype=np.float32
)

matriz_cf_final = csr_matrix(
    (
        valores_final,
        (
            indices_usuarios_final,
            colunas_final
        )
    ),
    shape=(
        len(usuarios_treino_final),
        len(catalogo_modelo)
    )
)

print("Formato:", matriz_cf_final.shape)
print("Interações armazenadas:", matriz_cf_final.nnz)

Formato: (5504668, 1030)
Interações armazenadas: 16016460


### 7.2 Similaridade colaborativa final

A partir da matriz de interações reconstruída com os dados disponíveis até o final de 2025, foi recalculada a similaridade de cosseno entre os cursos.

Nesta etapa, a matriz usuário × curso já utiliza diretamente as 1.030 posições do catálogo canônico. Assim, os 975 cursos com histórico comportamental possuem vetores de interação, enquanto os 55 cursos sem histórico permanecem representados por colunas sem interações.

A diagonal da matriz de similaridade é novamente zerada, uma vez que a similaridade de um curso consigo mesmo não participa da geração das recomendações.

In [77]:
inicio_cf_final = time.time()

similaridade_cf_final = cosine_similarity(
    matriz_cf_final.T,
    dense_output=True
).astype(np.float32)

np.fill_diagonal(
    similaridade_cf_final,
    0
)

fim_cf_final = time.time()

cursos_com_historico_final = (
    np.asarray(
        matriz_cf_final.sum(axis=0)
    ).ravel() > 0
)

print(
    "Formato:",
    similaridade_cf_final.shape
)

print(
    "Cursos com histórico:",
    cursos_com_historico_final.sum()
)

print(
    "Cursos sem histórico:",
    (~cursos_com_historico_final).sum()
)

print(
    "Similaridade máxima:",
    similaridade_cf_final.max()
)

print(
    f"Tempo: {fim_cf_final - inicio_cf_final:.2f} segundos"
)

Formato: (1030, 1030)
Cursos com histórico: 975
Cursos sem histórico: 55
Similaridade máxima: 0.9486833
Tempo: 15.79 segundos


### 7.3 Construção do conjunto de teste

Para a avaliação final, foram considerados os usuários que apresentavam histórico de interação anterior a 2026 e que, durante o período de teste, interagiram com pelo menos um curso ainda não presente em seu histórico.

Os cursos já realizados anteriormente pelo usuário não são considerados novos alvos de recomendação. Dessa forma, a avaliação busca verificar a capacidade do sistema de antecipar novos interesses observados entre 1º de janeiro e 2 de setembro de 2026.

Usuários que aparecem pela primeira vez em 2026 caracterizam cold-start de usuário e não participam desta avaliação personalizada, pois não possuem histórico a partir do qual o perfil individual possa ser construído.

In [78]:
pares_teste = con.execute(f"""
    WITH historico AS (
        SELECT DISTINCT
            codigo_pessoa,
            id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula < DATE '2026-01-01'
    ),

    interacoes_2026 AS (
        SELECT DISTINCT
            codigo_pessoa,
            id_curso_canonico
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        WHERE dt_matricula >= DATE '2026-01-01'
          AND dt_matricula <= DATE '2026-09-02'
    ),

    usuarios_com_historico AS (
        SELECT DISTINCT codigo_pessoa
        FROM historico
    )

    SELECT
        t.codigo_pessoa,
        t.id_curso_canonico
    FROM interacoes_2026 t

    INNER JOIN usuarios_com_historico u
        ON t.codigo_pessoa = u.codigo_pessoa

    LEFT JOIN historico h
        ON t.codigo_pessoa = h.codigo_pessoa
       AND t.id_curso_canonico = h.id_curso_canonico

    WHERE h.id_curso_canonico IS NULL
""").df()

print(
    "Pares relevantes no teste:",
    len(pares_teste)
)

print(
    "Usuários avaliáveis:",
    pares_teste["codigo_pessoa"].nunique()
)

print(
    "Cursos relevantes distintos:",
    pares_teste["id_curso_canonico"].nunique()
)

Pares relevantes no teste: 1572040
Usuários avaliáveis: 617448
Cursos relevantes distintos: 1029


Dos usuários com histórico anterior a 2026 que apresentaram interações durante o período de teste, 617.448 possuem pelo menos um novo curso que pode ser utilizado como alvo de avaliação. Foram identificados 1.572.040 pares relevantes usuário–curso, distribuídos por 1.029 cursos distintos.

Para esses usuários, foi reconstruído o histórico completo de cursos observados antes de 2026. Esse histórico é utilizado para gerar as recomendações dos componentes Content-Based e colaborativo.

In [79]:
usuarios_teste_df = (
    pares_teste[["codigo_pessoa"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

con.register(
    "usuarios_teste_df",
    usuarios_teste_df
)

historico_teste = con.execute(f"""
    SELECT DISTINCT
        i.codigo_pessoa,
        i.id_curso_canonico
    FROM read_parquet('{arquivo_interacoes.as_posix()}') i

    INNER JOIN usuarios_teste_df u
        ON i.codigo_pessoa = u.codigo_pessoa

    WHERE i.dt_matricula < DATE '2026-01-01'
""").df()

print(
    "Usuários:",
    historico_teste["codigo_pessoa"].nunique()
)

print(
    "Pares históricos:",
    len(historico_teste)
)

print(
    "Cursos distintos:",
    historico_teste["id_curso_canonico"].nunique()
)

Usuários: 617448
Pares históricos: 3546775
Cursos distintos: 975


### 7.4 Representação matricial do conjunto de teste

Para a avaliação final, o histórico anterior a 2026 e os novos cursos observados no período de teste foram representados por matrizes esparsas binárias.

As linhas correspondem aos 617.448 usuários avaliáveis e as colunas aos 1.030 cursos do catálogo canônico. A matriz de histórico registra os cursos previamente observados para cada usuário, enquanto a matriz de alvos registra os novos cursos com os quais o usuário interagiu em 2026.

Essa representação permite comparar diretamente as recomendações Top-10 produzidas pelo modelo com os itens efetivamente observados no período de teste.

In [83]:
usuarios_avaliacao_teste = (
    usuarios_teste_df["codigo_pessoa"]
    .tolist()
)

usuario_para_indice_teste = {
    usuario: indice
    for indice, usuario in enumerate(usuarios_avaliacao_teste)
}


# Matriz de histórico
linhas_historico_teste = (
    historico_teste["codigo_pessoa"]
    .map(usuario_para_indice_teste)
    .to_numpy()
)

colunas_historico_teste = (
    historico_teste["id_curso_canonico"]
    .map(id_para_indice)
    .to_numpy()
)

matriz_historico_teste = csr_matrix(
    (
        np.ones(len(historico_teste), dtype=np.float32),
        (
            linhas_historico_teste,
            colunas_historico_teste
        )
    ),
    shape=(
        len(usuarios_avaliacao_teste),
        len(catalogo_modelo)
    )
)


# Matriz de alvos
linhas_alvos_teste = (
    pares_teste["codigo_pessoa"]
    .map(usuario_para_indice_teste)
    .to_numpy()
)

colunas_alvos_teste = (
    pares_teste["id_curso_canonico"]
    .map(id_para_indice)
    .to_numpy()
)

matriz_alvos_teste = csr_matrix(
    (
        np.ones(len(pares_teste), dtype=np.float32),
        (
            linhas_alvos_teste,
            colunas_alvos_teste
        )
    ),
    shape=(
        len(usuarios_avaliacao_teste),
        len(catalogo_modelo)
    )
)


print("Histórico:", matriz_historico_teste.shape)
print("Pares no histórico:", matriz_historico_teste.nnz)

print()

print("Alvos:", matriz_alvos_teste.shape)
print("Pares relevantes:", matriz_alvos_teste.nnz)

Histórico: (617448, 1030)
Pares no histórico: 3546775

Alvos: (617448, 1030)
Pares relevantes: 1572040


### 7.5 Avaliação final dos modelos

Com o conjunto de teste preparado, os modelos Content-Based, Collaborative Filtering e Híbrido são avaliados sobre as interações observadas em 2026.

O componente colaborativo utiliza a matriz de similaridade reconstruída com todo o histórico disponível até o final de 2025. O componente baseado em conteúdo mantém a representação textual dos cursos definida anteriormente.

Para o modelo híbrido, utiliza-se α = 0,55, valor selecionado exclusivamente a partir do desempenho de NDCG@10 no conjunto de validação de 2025. Nenhum parâmetro é ajustado com base nos resultados de 2026.

Os três modelos são avaliados utilizando Recall@10, Hit Rate@10 e NDCG@10, mantendo o mesmo protocolo utilizado durante a validação.

In [84]:
def avaliar_modelos_teste(
    tamanho_lote=5000,
    k=10,
    alpha=0.55
):
    total_recall_content = 0.0
    total_hit_content = 0.0
    total_ndcg_content = 0.0

    total_recall_cf = 0.0
    total_hit_cf = 0.0
    total_ndcg_cf = 0.0

    total_recall_hibrido = 0.0
    total_hit_hibrido = 0.0
    total_ndcg_hibrido = 0.0

    n_usuarios = matriz_historico_teste.shape[0]

    descontos = 1 / np.log2(
        np.arange(2, k + 2)
    )

    for inicio in range(0, n_usuarios, tamanho_lote):

        fim = min(
            inicio + tamanho_lote,
            n_usuarios
        )

        historico_lote = matriz_historico_teste[inicio:fim]

        # Scores dos dois componentes
        scores_content = (
            historico_lote @ similaridade_content
        )

        scores_cf = (
            historico_lote @ similaridade_cf_final
        )

        # Cursos já vistos pelo usuário não podem ser recomendados
        linhas_vistas, colunas_vistas = historico_lote.nonzero()

        scores_content[
            linhas_vistas,
            colunas_vistas
        ] = 0

        scores_cf[
            linhas_vistas,
            colunas_vistas
        ] = 0

        # Normalização individual dos componentes
        max_content = scores_content.max(
            axis=1,
            keepdims=True
        )

        max_cf = scores_cf.max(
            axis=1,
            keepdims=True
        )

        max_content = np.where(
            max_content > 0,
            max_content,
            1
        )

        max_cf = np.where(
            max_cf > 0,
            max_cf,
            1
        )

        scores_content_norm = (
            scores_content / max_content
        )

        scores_cf_norm = (
            scores_cf / max_cf
        )

        # Combinação híbrida com alpha fixado na validação
        scores_hibrido = (
            alpha * scores_content_norm
            + (1 - alpha) * scores_cf_norm
        )

        # Para a avaliação individual dos modelos,
        # cursos vistos recebem -inf.
        scores_content[
            linhas_vistas,
            colunas_vistas
        ] = -np.inf

        scores_cf[
            linhas_vistas,
            colunas_vistas
        ] = -np.inf

        scores_hibrido[
            linhas_vistas,
            colunas_vistas
        ] = -np.inf

        # O CF puro só pode recomendar cursos
        # com histórico colaborativo até 2025.
        scores_cf[
            :,
            ~cursos_com_historico_final
        ] = -np.inf

        alvos_lote = (
            matriz_alvos_teste[inicio:fim]
            .toarray()
            .astype(bool)
        )

        qtd_relevantes = alvos_lote.sum(axis=1)

        resultados_lote = []

        for scores in [
            scores_content,
            scores_cf,
            scores_hibrido
        ]:

            top_k = np.argpartition(
                scores,
                -k,
                axis=1
            )[:, -k:]

            scores_top_k = np.take_along_axis(
                scores,
                top_k,
                axis=1
            )

            ordem = np.argsort(
                scores_top_k,
                axis=1
            )[:, ::-1]

            top_k = np.take_along_axis(
                top_k,
                ordem,
                axis=1
            )

            relevancia_top_k = np.take_along_axis(
                alvos_lote,
                top_k,
                axis=1
            )

            acertos = relevancia_top_k.sum(axis=1)

            recall = acertos / qtd_relevantes

            hit = (
                acertos > 0
            ).astype(float)

            dcg = (
                relevancia_top_k
                * descontos
            ).sum(axis=1)

            ideal = np.minimum(
                qtd_relevantes,
                k
            )

            idcg = np.array([
                descontos[:n].sum()
                for n in ideal
            ])

            ndcg = dcg / idcg

            resultados_lote.append(
                (
                    recall.sum(),
                    hit.sum(),
                    ndcg.sum()
                )
            )

        # Content-Based
        total_recall_content += resultados_lote[0][0]
        total_hit_content += resultados_lote[0][1]
        total_ndcg_content += resultados_lote[0][2]

        # Collaborative Filtering
        total_recall_cf += resultados_lote[1][0]
        total_hit_cf += resultados_lote[1][1]
        total_ndcg_cf += resultados_lote[1][2]

        # Híbrido
        total_recall_hibrido += resultados_lote[2][0]
        total_hit_hibrido += resultados_lote[2][1]
        total_ndcg_hibrido += resultados_lote[2][2]

    resultados = pd.DataFrame({
        "Modelo": [
            "Content-Based",
            "Collaborative Filtering",
            "Híbrido (α = 0,55)"
        ],
        "Recall@10": [
            total_recall_content / n_usuarios,
            total_recall_cf / n_usuarios,
            total_recall_hibrido / n_usuarios
        ],
        "Hit Rate@10": [
            total_hit_content / n_usuarios,
            total_hit_cf / n_usuarios,
            total_hit_hibrido / n_usuarios
        ],
        "NDCG@10": [
            total_ndcg_content / n_usuarios,
            total_ndcg_cf / n_usuarios,
            total_ndcg_hibrido / n_usuarios
        ]
    })

    return resultados

In [85]:
inicio_teste_final = time.time()

resultados_teste_final = avaliar_modelos_teste(
    tamanho_lote=5000,
    k=10,
    alpha=0.55
)

fim_teste_final = time.time()

display(resultados_teste_final)

print(
    f"Tempo total: {(fim_teste_final - inicio_teste_final) / 60:.2f} minutos"
)

,Modelo,Recall@10,Hit Rate@10,NDCG@10
0,Content-Based,0.095710,0.164864,0.065483
1,Collaborative Filtering,0.113944,0.189321,0.076082
2,"Híbrido (α = 0,55)",0.115061,0.191363,0.077997


Tempo total: 1.13 minutos


### 7.6 Análise dos resultados finais

No conjunto de teste temporal de 2026, o sistema híbrido apresentou Recall@10 de 0,1151, Hit Rate@10 de 0,1914 e NDCG@10 de 0,0780. Sob o mesmo protocolo de avaliação, esses valores foram superiores aos obtidos pelos componentes Content-Based e Collaborative Filtering avaliados isoladamente.

O Collaborative Filtering apresentou desempenho superior ao Content-Based nas três métricas, indicando que os padrões comportamentais extraídos das interações entre usuários e cursos possuem forte capacidade preditiva para os usuários que já apresentam histórico.

Entretanto, a combinação híbrida apresentou ganhos adicionais. O parâmetro α = 0,55, selecionado exclusivamente no conjunto de validação de 2025 a partir do NDCG@10, foi mantido sem alterações na avaliação de 2026. Dessa forma, o resultado do conjunto de teste não foi utilizado para reajustar a configuração do modelo.

Além da contribuição para o desempenho observado, a arquitetura híbrida possui uma vantagem estrutural importante. O componente colaborativo reconstruído até o final de 2025 possuía histórico para 975 dos 1.030 cursos canônicos. Para os 55 cursos restantes, sem histórico comportamental, o componente baseado em conteúdo ainda permite calcular relações a partir das características textuais disponíveis no catálogo.

A avaliação final considerou 617.448 usuários com histórico anterior a 2026 e pelo menos um novo curso observado no período de teste, totalizando 1.572.040 pares relevantes usuário–curso. As interações de 2026 utilizadas no teste compreendem o período disponível até 2 de setembro de 2026.

Os resultados devem ser interpretados considerando as limitações do experimento. A avaliação utiliza o catálogo atual como universo de candidatos, pois não foi possível reconstruir com precisão a disponibilidade histórica de cada curso. Além disso, usuários sem qualquer interação anterior a 2026 não participam da avaliação personalizada, uma vez que não possuem histórico para a construção de seus perfis.

Ainda assim, a separação cronológica entre treinamento, validação e teste permite avaliar o protótipo em uma situação mais próxima do uso real: informações anteriores são utilizadas para gerar recomendações que são comparadas com interações observadas posteriormente.

## 8. Limitações e possibilidades de evolução da POC

A presente Prova de Conceito teve como objetivo avaliar a viabilidade de uma arquitetura híbrida de recomendação utilizando informações textuais dos cursos e padrões de interação dos usuários. Embora os resultados obtidos indiquem a capacidade dos dois componentes de contribuir para as recomendações, algumas decisões adotadas nesta primeira implementação podem ser aperfeiçoadas em trabalhos futuros.

### 8.1 Representação textual e ponderação dos campos

No componente Content-Based, os campos textuais do catálogo foram concatenados para a construção da representação TF-IDF. Como estratégia heurística, o nome do curso e os eixos temáticos foram repetidos no texto combinado para aumentar sua influência na representação.

Embora adequada para uma primeira POC, essa estratégia não oferece controle explícito sobre a contribuição individual de cada grupo de atributos. Uma evolução possível consiste em construir representações separadas para campos como título, eixos temáticos e conteúdo descritivo e, posteriormente, combinar suas similaridades por meio de pesos definidos e avaliados no conjunto de validação.

Também pode ser avaliada a utilização de uma lista de stopwords adequada à língua portuguesa. A configuração atual utiliza parâmetros como `max_df` para reduzir a influência de termos extremamente frequentes, mas não realiza a remoção explícita de palavras funcionais da língua.

### 8.2 Representações semânticas

O TF-IDF representa principalmente relações baseadas na ocorrência dos termos e, portanto, possui capacidade limitada para identificar relações semânticas entre expressões diferentes.

Como evolução, podem ser avaliadas técnicas como Latent Semantic Analysis (LSA), por meio de redução de dimensionalidade com TruncatedSVD, ou representações baseadas em embeddings de linguagem. Essas abordagens podem permitir que cursos semanticamente relacionados sejam aproximados mesmo quando utilizam vocabulários diferentes em suas descrições.

### 8.3 Interpretação do feedback implícito

Nesta POC, cada relação consolidada entre usuário e curso é tratada como uma interação implícita binária. Os status das matrículas são preservados como informações de contexto, mas não recebem pesos positivos ou negativos.

Essa decisão evita assumir, por exemplo, que uma desistência representa necessariamente desinteresse pelo conteúdo. A análise exploratória identificou casos em que usuários posteriormente retornaram ao mesmo curso e o concluíram, indicando que o significado comportamental dos diferentes status não é necessariamente unívoco.

Como evolução, podem ser investigadas estratégias de ponderação do feedback implícito utilizando informações adicionais capazes de representar com maior segurança o nível de interesse ou engajamento do usuário, desde que os pesos adotados sejam definidos e avaliados de forma controlada.

### 8.4 Cold-start e disponibilidade histórica

O componente colaborativo depende da existência de interações anteriores e, portanto, apresenta limitações para usuários e cursos sem histórico. A abordagem híbrida reduz parte do problema relacionado aos itens, pois o componente Content-Based permite representar cursos a partir de seus atributos textuais.

Entretanto, usuários sem qualquer histórico anterior não podem receber recomendações personalizadas pelo método atualmente implementado. Estratégias baseadas em popularidade, contexto ou informações declaradas pelo próprio usuário podem ser investigadas para esse cenário.

Outra limitação está relacionada à disponibilidade histórica dos cursos. Como não foi possível reconstruir com precisão quais cursos estavam disponíveis em cada momento do período analisado, a avaliação utiliza o catálogo atual como universo de candidatos. Trabalhos futuros podem incorporar snapshots temporais do catálogo, caso essas informações se tornem disponíveis.